# Complete 379-region TVB experiment

**Primary research question**

How does increasing AD-like amyloid-linked inhibitory dysfunction affect stimulus-evoked transmission from bilateral primary auditory cortex to music-associated versus speech-associated cortical proxy subnetworks?

**Prespecified secondary question**

Do musical-semantic-task-associated and musical-episodic-task-associated parcel sets show different baseline-normalized transmission trajectories under the same perturbation?

This notebook runs the complete experiment on Google Colab. It uses:

- the public 379-region structural matrix used in the Stefanovski educational model;
- the matching 379-label Glasser plus subcortical parcel order from the BrainModes ADNI-TVB pipeline;
- the published Stefanovski amyloid-to-inhibitory-rate transformation;
- baseline, intermediate, and high AD-like perturbation strengths;
- identical pulse, 2 Hz, and 5 Hz probes delivered to bilateral A1;
- matched unstimulated simulations;
- full-field and local-dynamics-held-baseline analyses;
- topology/pathology-matched control subnetworks;
- coupling, stimulus-strength, spatial-placement, and integration-step checks;
- process-based multi-core execution with one native numerical thread per worker;
- a secondary semantic-versus-episodic proxy analysis mapped from the direct musical-memory contrasts reported by Platel et al. (2003).

**Central interpretation boundary:** this is a mechanistic simulation of one AD-related process. It is not a clinical HC/MCI/AD comparison, a model of disease progression, or a simulation of semantic or episodic memory.

## 1. What this experiment can and cannot establish

The public amyloid endpoint is an artificial surrogate constructed from noise and selected properties of ADNI-derived data. The intermediate condition is exactly halfway between the baseline and high endpoint in the model parameter \(b\). It is not an MCI brain.

The primary experiment can test whether the predeclared music- and speech-associated proxy subnetworks respond differently inside this specific amyloid-linked inhibitory model. The current high-resolution definitions contain 8 music-proxy parcels and 14 speech-proxy parcels. Their unequal sizes are retained because they represent different published anatomical systems; topology/pathology-matched null sets preserve each group size.

The secondary analysis keeps musical semantic and episodic associations separate:

- **Musical-semantic-task-associated proxy:** five HCP-MMP parcels mapped from the significant `semantic > episodic` cortical peaks in Platel et al. (2003).
- **Musical-episodic-task-associated proxy:** four HCP-MMP parcels mapped from the significant `episodic > semantic` cortical peaks in the same study.

These are task-associated peak-parcel proxies, not proven independent pathways. A newer matched-task study found substantial overlap rather than a reliable whole-brain separation between semantic and episodic retrieval, although that study did not use musical material. The split is therefore a prespecified secondary operationalization, not an anatomical fact.

The model receives periodic amplitude probes with no familiar/unfamiliar manipulation, encoding phase, delay, or recognition judgment. It can measure transmission into these parcel sets. It cannot instantiate or measure semantic familiarity, episodic recollection, or memory performance.

## 2. Colab setup

The installation is version-pinned for reproducibility. Colab normally already includes NumPy, pandas, Matplotlib, and SciPy.

In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "tvb": ("tvb-library==2.10.0", "tvb-data==3.0.0"),
    "pandas": ("pandas>=2.0",),
    "matplotlib": ("matplotlib>=3.7",),
    "scipy": ("scipy>=1.10",),
    "joblib": ("joblib>=1.4,<2.0",),
}

missing_specs = []
for import_name, specs in required.items():
    if importlib.util.find_spec(import_name) is None:
        missing_specs.extend(specs)

if missing_specs:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing_specs]
    )
else:
    print("Required packages are already available.")

In [ ]:
import gc
import hashlib
import json
import math
import os
import platform
import shutil
import time
import urllib.request
import warnings
from datetime import datetime, timezone
from pathlib import Path

# Process-level parallelism is the outer layer. Force every process to use
# one native numerical thread so n workers do not each create n BLAS threads.
NATIVE_THREAD_ENVIRONMENT_VARIABLES = (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "BLIS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
)
for variable_name in NATIVE_THREAD_ENVIRONMENT_VARIABLES:
    os.environ[variable_name] = "1"

os.environ.setdefault("TVB_USER_HOME", "/tmp/rise_tvb_user")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/rise_matplotlib")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from IPython.display import display
from joblib import Parallel, delayed, parallel_config
import tvb
from tvb.simulator.lab import (
    connectivity,
    coupling,
    equations,
    integrators,
    models,
    monitors,
    patterns,
    simulator,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
plt.style.use("seaborn-v0_8-whitegrid")

IS_COLAB = "google.colab" in sys.modules
WORK_DIR = (
    Path("/content/rise_tvb379")
    if IS_COLAB
    else Path.cwd() / "rise_tvb379_work"
)
DATA_DIR = WORK_DIR / "source_data"
WORK_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(
    {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "joblib": joblib.__version__,
        "tvb": getattr(tvb, "__version__", "unknown"),
        "colab": IS_COLAB,
        "work_dir": str(WORK_DIR),
    }
)


def _positive_environment_integer(variable_name):
    raw_value = os.environ.get(variable_name)
    if raw_value is None or not raw_value.strip():
        return None
    try:
        parsed = int(raw_value)
    except ValueError as error:
        raise ValueError(
            f"{variable_name} must be a positive integer, got {raw_value!r}."
        ) from error
    if parsed < 1:
        raise ValueError(
            f"{variable_name} must be a positive integer, got {parsed}."
        )
    return parsed


def detect_cpu_allocation():
    '''Return conservative CPU limits visible to this process.'''
    sources = {"joblib_cpu_count": max(1, int(joblib.cpu_count()))}

    affinity_function = getattr(os, "sched_getaffinity", None)
    if affinity_function is not None:
        try:
            affinity_count = len(affinity_function(0))
        except (OSError, TypeError, ValueError):
            affinity_count = 0
        if affinity_count > 0:
            sources["process_affinity"] = affinity_count

    # Respect common batch-scheduler allocations instead of using the
    # complete physical node when only part of it was requested.
    for variable_name in (
        "SLURM_CPUS_PER_TASK",
        "NSLOTS",
        "PBS_NP",
        "LSB_DJOB_NUMPROC",
    ):
        value = _positive_environment_integer(variable_name)
        if value is not None:
            sources[variable_name] = value

    return sources


class SimpleProgress:
    def __init__(self, total, description):
        self.total = int(total)
        self.description = str(description)
        self.count = 0
        self.report_every = max(1, self.total // 10)
        self.next_report = self.report_every
        print(f"{self.description}: 0/{self.total}")

    def update(self, amount=1):
        self.count += int(amount)
        if self.count >= self.next_report or self.count >= self.total:
            print(
                f"{self.description}: "
                f"{min(self.count, self.total)}/{self.total}"
            )
            while self.next_report <= self.count:
                self.next_report += self.report_every

    def close(self):
        if self.count != self.total:
            print(f"{self.description}: stopped at {self.count}/{self.total}")


def progress_iter(values, description):
    values = list(values)
    progress = SimpleProgress(len(values), description)
    for value in values:
        yield value
        progress.update()
    progress.close()


def initialize_parallel_worker(work_dir):
    '''Give each spawned worker private TVB and Matplotlib runtime paths.'''
    worker_root = (
        Path(work_dir)
        / ".parallel_runtime"
        / f"worker_{os.getpid()}"
    )
    tvb_home = worker_root / "tvb"
    matplotlib_home = worker_root / "matplotlib"
    tvb_home.mkdir(parents=True, exist_ok=True)
    matplotlib_home.mkdir(parents=True, exist_ok=True)
    os.environ["TVB_USER_HOME"] = str(tvb_home)
    os.environ["MPLCONFIGDIR"] = str(matplotlib_home)
    for variable_name in NATIVE_THREAD_ENVIRONMENT_VARIABLES:
        os.environ[variable_name] = "1"
    if os.environ.get("RISE_DISABLE_LOKY_PSUTIL", "").strip() == "1":
        # Workaround for restricted PID namespaces where psutil cannot see
        # the worker's own /proc entry. Normal Colab/VM runs leave this off.
        from joblib.externals.loky import process_executor
        process_executor._USE_PSUTIL = False


def run_parallel_jobs(
    worker_function,
    job_payloads,
    shared_args,
    description,
):
    '''Run independent jobs in loky worker processes and stream completions.'''
    jobs = list(job_payloads)
    if not jobs:
        return []

    progress = SimpleProgress(len(jobs), description)
    outcomes = []
    try:
        if PARALLEL_WORKERS == 1:
            for job in jobs:
                outcomes.append(worker_function(job, *shared_args))
                progress.update()
            return outcomes

        memmap_dir = WORK_DIR / ".joblib_memmap"
        memmap_dir.mkdir(parents=True, exist_ok=True)
        with parallel_config(
            backend="loky",
            n_jobs=PARALLEL_WORKERS,
            inner_max_num_threads=1,
            initializer=initialize_parallel_worker,
            initargs=(str(WORK_DIR),),
            idle_worker_timeout=900,
            temp_folder=str(memmap_dir),
            max_nbytes="512K",
            mmap_mode="r",
        ):
            outcome_stream = Parallel(
                return_as="generator_unordered",
                batch_size=1,
                pre_dispatch="2*n_jobs",
            )(
                delayed(worker_function)(job, *shared_args)
                for job in jobs
            )
            for outcome in outcome_stream:
                outcomes.append(outcome)
                progress.update()
        return outcomes
    finally:
        progress.close()


## 3. Run configuration

`final` is the default and runs the full experiment. For an initial technical check, change it to `pilot`. The `smoke` mode exists for automated notebook validation and is not adequate for scientific interpretation.

The notebook uses **process-based CPU parallelism** through joblib's `loky` backend. This is deliberate: the TVB simulations are CPU-bound, and ordinary Python threads would not reliably use multiple cores. One condition-seed block is assigned to each worker, and every worker is limited to one BLAS/OpenMP thread.

By default, the notebook uses all CPUs available to the current Colab, Codespace, virtual machine, or scheduler allocation. To request fewer workers, set `RISE_N_WORKERS` before running this cell, for example:

```python
os.environ["RISE_N_WORKERS"] = "8"
```

Requests above the detected allocation are clamped rather than oversubscribing the machine. Numerical seeds test sensitivity to initial conditions. They are repeated model runs, not participants or independent biological samples.

In [ ]:
RUN_MODE = os.environ.get("RISE_RUN_MODE", "final").strip().lower()
if RUN_MODE not in {"smoke", "pilot", "final"}:
    raise ValueError("RUN_MODE must be 'smoke', 'pilot', or 'final'.")

MODE_CONFIG = {
    "smoke": {
        "seeds": [11],
        "severities": [0.0, 1.0],
        "calibration_couplings": [60.0],
        "matched_null_sets": 40,
        "spatial_shuffles": 1,
        "sensitivity_scenarios": [
            {"scenario": "G30", "global_coupling": 30.0, "input_peak": 0.02}
        ],
    },
    "pilot": {
        "seeds": [11, 23],
        "severities": [0.0, 0.5, 1.0],
        "calibration_couplings": [30.0, 60.0, 100.0],
        "matched_null_sets": 200,
        "spatial_shuffles": 2,
        "sensitivity_scenarios": [
            {"scenario": "G30", "global_coupling": 30.0, "input_peak": 0.02},
            {"scenario": "G100", "global_coupling": 100.0, "input_peak": 0.02},
        ],
    },
    "final": {
        "seeds": [11, 23, 37, 53, 71],
        "severities": [0.0, 0.5, 1.0],
        "calibration_couplings": [10.0, 30.0, 60.0, 100.0, 200.0, 300.0],
        "matched_null_sets": 500,
        "spatial_shuffles": 100,
        "sensitivity_scenarios": [
            {"scenario": "G30", "global_coupling": 30.0, "input_peak": 0.02},
            {"scenario": "G100", "global_coupling": 100.0, "input_peak": 0.02},
            {"scenario": "input_0.01", "global_coupling": 60.0, "input_peak": 0.01},
            {"scenario": "input_0.04", "global_coupling": 60.0, "input_peak": 0.04},
        ],
    },
}
CFG = MODE_CONFIG[RUN_MODE]

N_REGIONS = 379
MAIN_GLOBAL_COUPLING = 60.0
MAIN_INPUT_PEAK_PER_MS = 0.02
MAIN_DT_MS = 0.5
REFERENCE_DT_MS = 0.25
MONITOR_PERIOD_MS = 2.0

STIMULUS_ONSET_MS = 2500.0
SIMULATION_MS = 6500.0
PERIODIC_ANALYSIS_START_MS = 3500.0
PULSE_WIDTH_MS = 100.0
PULSE_ANALYSIS_END_MS = 3500.0

PROBES = ("pulse", "2Hz", "5Hz")
PERIODIC_PROBES = ("2Hz", "5Hz")
SEVERITY_LABELS = {
    0.0: "Baseline",
    0.5: "Intermediate AD-like perturbation",
    1.0: "High AD-like perturbation",
}
DOWNLOAD_RESULTS_AT_END = False

CPU_ALLOCATION_SOURCES = detect_cpu_allocation()
AVAILABLE_CPU_COUNT = min(CPU_ALLOCATION_SOURCES.values())
REQUESTED_PARALLEL_WORKERS = _positive_environment_integer(
    "RISE_N_WORKERS"
)
if REQUESTED_PARALLEL_WORKERS is None:
    PARALLEL_WORKERS = AVAILABLE_CPU_COUNT
else:
    PARALLEL_WORKERS = min(
        REQUESTED_PARALLEL_WORKERS,
        AVAILABLE_CPU_COUNT,
    )
    if REQUESTED_PARALLEL_WORKERS > AVAILABLE_CPU_COUNT:
        print(
            "Requested RISE_N_WORKERS exceeds the detected CPU "
            f"allocation; using {AVAILABLE_CPU_COUNT} workers."
        )
PARALLEL_BACKEND = "joblib-loky-processes"
NATIVE_THREADS_PER_WORKER = 1

RESULTS_DIR = WORK_DIR / f"results_{RUN_MODE}"
FIGURE_DIR = RESULTS_DIR / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_MODE:", RUN_MODE)
print("Numerical seeds:", CFG["seeds"])
print("Main severities:", CFG["severities"])
print("CPU allocation sources:", CPU_ALLOCATION_SOURCES)
print("Parallel worker processes:", PARALLEL_WORKERS)
print("Native numerical threads per worker:", NATIVE_THREADS_PER_WORKER)
print("Results directory:", RESULTS_DIR)

## 4. Download pinned source files and verify their hashes

Every required file is pinned to a specific Git commit and checked against a known SHA-256 hash. A failed hash check stops the notebook rather than silently running with a different matrix or label order.

In [ ]:
EDUCASE_COMMIT = "659d4fcbf58d74867fa9d10a874deac854532ee1"
PIPELINE_COMMIT = "8be09e33e1131ed2f0764506940e6172de275285"

SOURCE_SPECS = {
    "structural_connectivity": {
        "filename": "avg_healthy_normSC_mod.txt",
        "url": (
            "https://raw.githubusercontent.com/BrainModes/"
            "TVB_EducaseAD_molecular_pathways_TVB/"
            f"{EDUCASE_COMMIT}/avg_healthy_normSC_mod.txt"
        ),
        "sha256": "141fc993c84bde0b2f0ee0280ce1ccc47e1731ddcbf37845a4eef38dad9fa562",
    },
    "ad_left_cortex": {
        "filename": "AD_LH.txt",
        "url": (
            "https://raw.githubusercontent.com/BrainModes/"
            "TVB_EducaseAD_molecular_pathways_TVB/"
            f"{EDUCASE_COMMIT}/_AD/AD_LH.txt"
        ),
        "sha256": "566e770e93f50d3378a0cf7d2dc8b1fa5af9475ca432463acf7bc2b5507907af",
    },
    "ad_right_cortex": {
        "filename": "AD_RH.txt",
        "url": (
            "https://raw.githubusercontent.com/BrainModes/"
            "TVB_EducaseAD_molecular_pathways_TVB/"
            f"{EDUCASE_COMMIT}/_AD/AD_RH.txt"
        ),
        "sha256": "4f42c31c08e6d191d415953f763889f816d9c2443d115612f0c076cfd5b2f129",
    },
    "ad_subcortical": {
        "filename": "AD_subcortical.txt",
        "url": (
            "https://raw.githubusercontent.com/BrainModes/"
            "TVB_EducaseAD_molecular_pathways_TVB/"
            f"{EDUCASE_COMMIT}/_AD/AD_subcortical.txt"
        ),
        "sha256": "f28926c7955db2c2762b5ba032f0bbde770a0da3d3705dd037a746382506d824",
    },
    "region_labels": {
        "filename": "region_labels.txt",
        "url": (
            "https://raw.githubusercontent.com/BrainModes/"
            f"ADNI-TVB-pipeline/{PIPELINE_COMMIT}/misc_files/region_labels.txt"
        ),
        "sha256": "f9688592130a034210b482a0556fdc14383eaaf578bef39d2ddba072537e3484",
    },
}

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def fetch_verified(name, spec):
    destination = DATA_DIR / spec["filename"]
    local_cache = os.environ.get("RISE_DATA_CACHE")
    cache_candidate = (
        Path(local_cache) / spec["filename"] if local_cache else None
    )

    if destination.exists() and sha256_file(destination) == spec["sha256"]:
        source_used = "existing verified file"
    elif (
        cache_candidate is not None
        and cache_candidate.exists()
        and sha256_file(cache_candidate) == spec["sha256"]
    ):
        shutil.copyfile(cache_candidate, destination)
        source_used = "verified local validation cache"
    else:
        print(f"Downloading {name}...")
        with urllib.request.urlopen(spec["url"], timeout=120) as response:
            destination.write_bytes(response.read())
        source_used = spec["url"]

    actual = sha256_file(destination)
    if actual != spec["sha256"]:
        raise RuntimeError(
            f"SHA-256 mismatch for {name}. Expected {spec['sha256']}, got {actual}."
        )
    return {
        "source": name,
        "path": str(destination),
        "sha256": actual,
        "retrieved_from": source_used,
    }

source_manifest_df = pd.DataFrame(
    [fetch_verified(name, spec) for name, spec in SOURCE_SPECS.items()]
)
display(source_manifest_df)

## 5. Load and validate the 379-region alignment

These checks are scientific safeguards, not cosmetic tests. The amyloid vector, structural matrix, and parcel labels must have the exact same order. The structural matrix is intentionally left in its published form. It is not symmetrized or renormalized for simulation.

In [ ]:
WEIGHTS = np.loadtxt(DATA_DIR / SOURCE_SPECS["structural_connectivity"]["filename"])
LABELS = np.array(
    [
        line.strip()
        for line in (
            DATA_DIR / SOURCE_SPECS["region_labels"]["filename"]
        ).read_text().splitlines()
        if line.strip()
    ]
)
AD_AMYLOID = np.concatenate(
    [
        np.loadtxt(DATA_DIR / SOURCE_SPECS["ad_left_cortex"]["filename"]),
        np.loadtxt(DATA_DIR / SOURCE_SPECS["ad_right_cortex"]["filename"]),
        np.loadtxt(DATA_DIR / SOURCE_SPECS["ad_subcortical"]["filename"]),
    ]
)

hard_checks = {
    "SC is 379 x 379": WEIGHTS.shape == (N_REGIONS, N_REGIONS),
    "379 unique labels": len(LABELS) == N_REGIONS
    and len(set(LABELS.tolist())) == N_REGIONS,
    "379 amyloid values": AD_AMYLOID.shape == (N_REGIONS,),
    "SC values finite": np.isfinite(WEIGHTS).all(),
    "SC values nonnegative": (WEIGHTS >= 0).all(),
    "amyloid values finite": np.isfinite(AD_AMYLOID).all(),
    "left cortex occupies 0:180": all(
        label.startswith("L_") for label in LABELS[:180]
    ),
    "right cortex occupies 180:360": all(
        label.startswith("R_") for label in LABELS[180:360]
    ),
    "brainstem is final parcel": LABELS[-1] == "Brainstem",
}
failed_checks = [name for name, passed in hard_checks.items() if not passed]
if failed_checks:
    raise RuntimeError("Input validation failed: " + "; ".join(failed_checks))

LABEL_TO_INDEX = {label: index for index, label in enumerate(LABELS)}
expected_anchors = {
    "L_A1": 23,
    "R_A1": 203,
    "L_6ma": 43,
    "R_6ma": 223,
    "L_24dd": 39,
    "R_24dd": 219,
    "L_STSdp": 128,
    "R_STSdp": 308,
    "L_44": 73,
    "R_44": 253,
}
anchor_failures = {
    label: (LABEL_TO_INDEX.get(label), expected_index)
    for label, expected_index in expected_anchors.items()
    if LABEL_TO_INDEX.get(label) != expected_index
}
if anchor_failures:
    raise RuntimeError(f"Parcel-order anchor check failed: {anchor_failures}")

relative_asymmetry = float(
    np.linalg.norm(WEIGHTS - WEIGHTS.T) / np.linalg.norm(WEIGHTS)
)
data_quality_df = pd.DataFrame(
    {
        "check": list(hard_checks.keys())
        + [
            "SC minimum",
            "SC maximum",
            "SC relative asymmetry",
            "amyloid minimum",
            "amyloid maximum",
            "amyloid mean",
        ],
        "result": list(hard_checks.values())
        + [
            float(WEIGHTS.min()),
            float(WEIGHTS.max()),
            relative_asymmetry,
            float(AD_AMYLOID.min()),
            float(AD_AMYLOID.max()),
            float(AD_AMYLOID.mean()),
        ],
    }
)
display(data_quality_df)
print(
    "The nonzero asymmetry is a property of the published matrix. "
    "The simulation preserves it."
)

## 6. Predeclare the stimulation and target parcels

All parcel selections are fixed before the disease-perturbation results are examined.

### Primary comparison

- Bilateral `A1` receives the stimulus.
- The music proxy contains bilateral `TA2`, `STGa`, `6ma`, and `24dd`.
- The speech proxy contains bilateral `PSL`, `PFcm`, `STSdp`, and `TPOJ1`, plus left `44`, `45`, `47l`, `FOP4`, `8C`, and `SCEF`.
- Belt, parabelt, association-auditory, dorsal-speech, ventral-speech, and broader music-context groups are retained as separate diagnostics.

### Secondary musical-memory comparison

Platel et al. (2003) directly contrasted musical semantic-familiarity judgments with delayed musical recognition. The significant cortical peak coordinates from the two direct contrasts were mapped to HCP-MMP parcels using a volumetric HCP-MMP1 reference. The cerebellar semantic peak was excluded because this 379-node model has no cerebellar parcels.

The resulting secondary sets are:

- **Semantic-task-associated:** `R_9m`, `L_25`, `R_TE1a`, `L_TF`, `L_STSda`
- **Episodic-task-associated:** `R_IP1`, `R_PCV`, `R_11l`, `R_8Av`

The mapping table below records every source coordinate, selected parcel, and mapping distance. `R_11l` was selected as the nearest parcel consistent with the paper's reported BA11 anatomy; the coordinate falls on a volumetric atlas boundary.

Slattery et al. (2019) provides independent AD relevance: in an AD cohort, musical semantic behavior was relatively preserved while incidental musical episodic behavior was impaired, with disease-associated activation differences in right inferior frontal cortex versus precuneus/posterior cingulate cortex. That evidence motivates the secondary question but is not used to relabel these periodic probes as memory tasks.

In [ ]:
def bilateral(*parcel_names):
    """Convert HCP parcel names into alternating left/right labels."""
    return tuple(
        label
        for parcel in parcel_names
        for label in (f"L_{parcel}", f"R_{parcel}")
    )


def ordered_union(*groups):
    """Combine label groups while preserving order and removing duplicates."""
    return tuple(
        dict.fromkeys(
            label
            for group in groups
            for label in group
        )
    )


# Each mapping is explicitly labeled as stimulus, primary target,
# shared relay, or diagnostic-only context.
ROI_GROUPS = {
    "a1_input": {
        "labels": bilateral("A1"),
        "analysis_role": "stimulus",
        "interpretation": (
            "Bilateral primary auditory cortex receiving the external input."
        ),
    },

    # Shared auditory trunk used by both speech and music.
    "shared_early_auditory_relay": {
        "labels": bilateral("52", "MBelt", "LBelt", "RI"),
        "analysis_role": "shared_relay",
        "interpretation": (
            "Early auditory belt and retroinsular relay shared by both branches."
        ),
    },
    "shared_parabelt": {
        "labels": bilateral("PBelt"),
        "analysis_role": "shared_relay",
        "interpretation": (
            "Available HCP parabelt parcel. HCP-MMP does not separately "
            "represent rostral and caudal parabelt."
        ),
    },
    "shared_auditory_association": {
        "labels": bilateral("A4", "A5"),
        "analysis_role": "shared_relay",
        "interpretation": (
            "Nonprimary auditory association cortex downstream of parabelt."
        ),
    },

    # Primary music-associated comparison targets.
    "diagram_music_temporal_proxy": {
        "labels": bilateral("TA2", "STGa"),
        "analysis_role": "primary_music",
        "interpretation": (
            "Planum-polare/anterior auditory proxy from the pathway diagram. "
            "This is not parcel-level proof of music selectivity."
        ),
    },
    "music_memory_core_proxy": {
        "labels": bilateral("6ma", "24dd"),
        "analysis_role": "primary_music",
        "interpretation": (
            "Parcel approximations of ventral pre-SMA and caudal anterior "
            "cingulate musical-memory regions."
        ),
    },

    # Secondary musical-memory task proxies. These are mapped from the
    # direct semantic-versus-episodic PET contrasts in Platel et al. (2003).
    "music_semantic_task_associated": {
        "labels": (
            "R_9m",
            "L_25",
            "R_TE1a",
            "L_TF",
            "L_STSda",
        ),
        "analysis_role": "secondary_music_memory",
        "interpretation": (
            "HCP-MMP parcels mapped from cortical peaks in the Platel et al. "
            "semantic-greater-than-episodic musical-memory contrast."
        ),
    },
    "music_episodic_task_associated": {
        "labels": (
            "R_IP1",
            "R_PCV",
            "R_11l",
            "R_8Av",
        ),
        "analysis_role": "secondary_music_memory",
        "interpretation": (
            "HCP-MMP parcels mapped from cortical peaks in the Platel et al. "
            "episodic-greater-than-semantic musical-memory contrast."
        ),
    },

    # Music-related regions from the diagram that should be inspected
    # separately instead of being allowed to dominate the primary average.
    "music_anterior_temporal_context": {
        "labels": bilateral("TGd", "TGv", "TE1a", "TE2a"),
        "analysis_role": "diagnostic_only",
        "interpretation": (
            "Anterior temporal semantic and conceptual context."
        ),
    },
    "music_frontoparietal_context": {
        "labels": bilateral(
            "10r",
            "47l",
            "a9-46v",
            "46",
            "8Ad",
            "8Av",
            "PFm",
            "PGi",
            "PGs",
        ),
        "analysis_role": "diagnostic_only",
        "interpretation": (
            "Diagram-mapped ventrolateral/dorsolateral prefrontal and "
            "inferior-parietal context."
        ),
    },
    "music_medial_temporal_context": {
        "labels": (
            "Left-Hippocampus",
            "Right-Hippocampus",
            "Left-Amygdala",
            "Right-Amygdala",
        ),
        "analysis_role": "diagnostic_only",
        "interpretation": (
            "Medial-temporal context from the diagram. The Jansen-Rit model "
            "does not implement memory encoding or retrieval."
        ),
    },

    # Primary speech-associated comparison targets.
    "speech_posterior_temporal": {
        "labels": bilateral("PSL", "PFcm", "STSdp", "TPOJ1"),
        "analysis_role": "primary_speech",
        "interpretation": (
            "Bilateral posterior temporal and perisylvian auditory-language "
            "targets."
        ),
    },
    "speech_left_frontal": {
        "labels": (
            "L_44",
            "L_45",
            "L_47l",
            "L_FOP4",
            "L_8C",
            "L_SCEF",
        ),
        "analysis_role": "primary_speech",
        "interpretation": (
            "Left-lateralized inferior and medial frontal language targets."
        ),
    },

    # Full dorsal and ventral speech routes from the diagram.
    "speech_dorsal_left": {
        "labels": (
            "L_PSL",
            "L_PFcm",
            "L_TPOJ1",
            "L_PF",
            "L_PFm",
            "L_PGi",
            "L_PGs",
            "L_PGp",
            "L_55b",
            "L_6r",
            "L_4",
            "L_44",
            "L_FOP4",
            "L_8C",
            "L_SCEF",
        ),
        "analysis_role": "diagnostic_only",
        "interpretation": (
            "Left temporoparietal-to-premotor/articulatory dorsal stream."
        ),
    },
    "speech_ventral": {
        "labels": ordered_union(
            bilateral(
                "STSda",
                "STSdp",
                "STSva",
                "STSvp",
                "TE1a",
                "TE1m",
                "TE1p",
                "TE2a",
                "TE2p",
                "TGd",
                "TGv",
            ),
            ("L_45", "L_47l"),
        ),
        "analysis_role": "diagnostic_only",
        "interpretation": (
            "Bilateral temporal semantic stream with left inferior-frontal "
            "outputs."
        ),
    },
}


# Validate every hard-coded label against the exact loaded 379-entry order.
ALL_ROI_LABELS = ordered_union(
    *(spec["labels"] for spec in ROI_GROUPS.values())
)

missing_roi_labels = sorted(set(ALL_ROI_LABELS) - set(LABELS))
if missing_roi_labels:
    raise RuntimeError(
        f"ROI labels absent from the 379-region order: {missing_roi_labels}"
    )

for group_name, specification in ROI_GROUPS.items():
    group_labels = specification["labels"]
    if len(group_labels) != len(set(group_labels)):
        raise RuntimeError(
            f"Duplicate label inside ROI group {group_name!r}."
        )


A1_LABELS = ROI_GROUPS["a1_input"]["labels"]

MUSIC_LABELS = ordered_union(
    *(
        spec["labels"]
        for spec in ROI_GROUPS.values()
        if spec["analysis_role"] == "primary_music"
    )
)

SPEECH_LABELS = ordered_union(
    *(
        spec["labels"]
        for spec in ROI_GROUPS.values()
        if spec["analysis_role"] == "primary_speech"
    )
)

# The primary groups are non-overlapping. Diagnostic groups may overlap
# because the diagram contains genuinely shared higher-order regions.
primary_overlap = sorted(set(MUSIC_LABELS) & set(SPEECH_LABELS))
if primary_overlap:
    raise RuntimeError(
        f"Primary music and speech proxies overlap: {primary_overlap}"
    )


A1_INDICES = np.array(
    [LABEL_TO_INDEX[label] for label in A1_LABELS],
    dtype=int,
)
MUSIC_INDICES = np.array(
    [LABEL_TO_INDEX[label] for label in MUSIC_LABELS],
    dtype=int,
)
SPEECH_INDICES = np.array(
    [LABEL_TO_INDEX[label] for label in SPEECH_LABELS],
    dtype=int,
)

SEMANTIC_MEMORY_LABELS = ROI_GROUPS[
    "music_semantic_task_associated"
]["labels"]
EPISODIC_MEMORY_LABELS = ROI_GROUPS[
    "music_episodic_task_associated"
]["labels"]

secondary_overlap = sorted(
    set(SEMANTIC_MEMORY_LABELS) & set(EPISODIC_MEMORY_LABELS)
)
if secondary_overlap:
    raise RuntimeError(
        "Secondary musical-memory proxy sets overlap: "
        f"{secondary_overlap}"
    )

SEMANTIC_MEMORY_INDICES = np.array(
    [LABEL_TO_INDEX[label] for label in SEMANTIC_MEMORY_LABELS],
    dtype=int,
)
EPISODIC_MEMORY_INDICES = np.array(
    [LABEL_TO_INDEX[label] for label in EPISODIC_MEMORY_LABELS],
    dtype=int,
)

# Used to prevent declared pathway parcels from appearing in matched-null sets.
ALL_DECLARED_INDICES = np.array(
    sorted(
        {
            LABEL_TO_INDEX[label]
            for label in ALL_ROI_LABELS
        }
    ),
    dtype=int,
)

# Separate counterfactual masks prevent one analysis from silently
# changing the local dynamics of the other analysis's target parcels.
PRIMARY_COUNTERFACTUAL_FIXED_INDICES = np.array(
    sorted(
        set(A1_INDICES.tolist())
        | set(MUSIC_INDICES.tolist())
        | set(SPEECH_INDICES.tolist())
    ),
    dtype=int,
)
MEMORY_COUNTERFACTUAL_FIXED_INDICES = np.array(
    sorted(
        set(A1_INDICES.tolist())
        | set(SEMANTIC_MEMORY_INDICES.tolist())
        | set(EPISODIC_MEMORY_INDICES.tolist())
    ),
    dtype=int,
)


# Published SPM99 peak coordinates and their HCP-MMP parcel mapping.
# Distances were measured in a volumetric HCP-MMP1 reference. These
# coordinates are provenance for the operational proxy, not a runtime atlas.
MUSIC_MEMORY_PEAK_MAPPINGS = [
    {
        "source_contrast": "semantic > episodic",
        "reported_region": "bilateral medial frontal cortex (BA 11/10)",
        "spm99_x": 0,
        "spm99_y": 60,
        "spm99_z": 10,
        "hcp_label": "R_9m",
        "mapping_distance_mm": 1.0,
        "mapping_note": "nearest volumetric HCP-MMP parcel",
    },
    {
        "source_contrast": "semantic > episodic",
        "reported_region": "bilateral medial frontal cortex (BA 11/10)",
        "spm99_x": -4,
        "spm99_y": 18,
        "spm99_z": -18,
        "hcp_label": "L_25",
        "mapping_distance_mm": 0.0,
        "mapping_note": "coordinate falls inside parcel",
    },
    {
        "source_contrast": "semantic > episodic",
        "reported_region": "right middle temporal gyrus (BA 21)",
        "spm99_x": 56,
        "spm99_y": 4,
        "spm99_z": -24,
        "hcp_label": "R_TE1a",
        "mapping_distance_mm": 0.0,
        "mapping_note": "coordinate falls inside parcel",
    },
    {
        "source_contrast": "semantic > episodic",
        "reported_region": "left inferior/middle temporal gyri (BA 20/21)",
        "spm99_x": -48,
        "spm99_y": -26,
        "spm99_z": -22,
        "hcp_label": "L_TF",
        "mapping_distance_mm": 0.0,
        "mapping_note": "coordinate falls inside parcel",
    },
    {
        "source_contrast": "semantic > episodic",
        "reported_region": "left inferior/middle temporal gyri (BA 20/21)",
        "spm99_x": -54,
        "spm99_y": -2,
        "spm99_z": -18,
        "hcp_label": "L_STSda",
        "mapping_distance_mm": 0.0,
        "mapping_note": "coordinate falls inside parcel",
    },
    {
        "source_contrast": "episodic > semantic",
        "reported_region": "right precuneus/parietal cortex (BA 7/19)",
        "spm99_x": 36,
        "spm99_y": -66,
        "spm99_z": 38,
        "hcp_label": "R_IP1",
        "mapping_distance_mm": 2.0,
        "mapping_note": "nearest volumetric HCP-MMP parcel",
    },
    {
        "source_contrast": "episodic > semantic",
        "reported_region": "precuneus (BA 7)",
        "spm99_x": 4,
        "spm99_y": -56,
        "spm99_z": 42,
        "hcp_label": "R_PCV",
        "mapping_distance_mm": 0.0,
        "mapping_note": "coordinate falls inside parcel",
    },
    {
        "source_contrast": "episodic > semantic",
        "reported_region": "right superior frontal gyrus (BA 11)",
        "spm99_x": 34,
        "spm99_y": 52,
        "spm99_z": -14,
        "hcp_label": "R_11l",
        "mapping_distance_mm": 1.0,
        "mapping_note": (
            "nearest parcel consistent with reported BA11 anatomy; "
            "volumetric boundary"
        ),
    },
    {
        "source_contrast": "episodic > semantic",
        "reported_region": "right middle frontal gyrus (BA 8/9)",
        "spm99_x": 38,
        "spm99_y": 12,
        "spm99_z": 44,
        "hcp_label": "R_8Av",
        "mapping_distance_mm": 1.0,
        "mapping_note": "nearest volumetric HCP-MMP parcel",
    },
]
music_memory_peak_mapping_df = pd.DataFrame(MUSIC_MEMORY_PEAK_MAPPINGS)

expected_semantic_mapping = set(SEMANTIC_MEMORY_LABELS)
expected_episodic_mapping = set(EPISODIC_MEMORY_LABELS)
actual_semantic_mapping = set(
    music_memory_peak_mapping_df.loc[
        music_memory_peak_mapping_df["source_contrast"]
        == "semantic > episodic",
        "hcp_label",
    ]
)
actual_episodic_mapping = set(
    music_memory_peak_mapping_df.loc[
        music_memory_peak_mapping_df["source_contrast"]
        == "episodic > semantic",
        "hcp_label",
    ]
)
if actual_semantic_mapping != expected_semantic_mapping:
    raise RuntimeError("Semantic peak mapping and ROI labels disagree.")
if actual_episodic_mapping != expected_episodic_mapping:
    raise RuntimeError("Episodic peak mapping and ROI labels disagree.")


roi_rows = []
for group_name, specification in ROI_GROUPS.items():
    for label in specification["labels"]:
        roi_rows.append(
            {
                "network": group_name,
                "analysis_role": specification["analysis_role"],
                "label": label,
                "zero_based_index": LABEL_TO_INDEX[label],
                "interpretation": specification["interpretation"],
            }
        )

roi_definition_df = pd.DataFrame(roi_rows)

print("Primary music parcels:", len(MUSIC_LABELS))
print("Primary speech parcels:", len(SPEECH_LABELS))
print("Secondary semantic-task-associated parcels:", len(SEMANTIC_MEMORY_LABELS))
print("Secondary episodic-task-associated parcels:", len(EPISODIC_MEMORY_LABELS))
print("Unique declared pathway parcels:", len(ALL_DECLARED_INDICES))
display(roi_definition_df)
display(music_memory_peak_mapping_df)

## 7. Construct the AD-like inhibitory perturbation

Stefanovski et al. mapped regional amyloid burden to the Jansen-Rit inhibitory rate \(b=1/\tau_i\). The same published sigmoid is used here.

- **Baseline:** \(b=0.07\) at every parcel.
- **Intermediate AD-like perturbation:** halfway from baseline to the transformed high endpoint in \(b\)-space.
- **High AD-like perturbation:** the full transformed public AD surrogate vector.

The intermediate value is an experimental perturbation level. It is not labeled MCI.

In [ ]:
def transform_amyloid_to_b(
    amyloid,
    max_val=0.05,
    min_val=0.02,
    amyloid_max=2.65,
    amyloid_offset=1.4,
):
    amyloid = np.asarray(amyloid, dtype=float)
    x0 = (amyloid_max - amyloid_offset) / 2.0 + amyloid_offset
    k = (
        np.log(max_val / ((min_val + 0.001) - min_val) - 1.0)
        / (amyloid_max - x0)
    )
    return max_val / (1.0 + np.exp(k * (amyloid - x0))) + min_val

BASELINE_B = np.full(N_REGIONS, 0.07, dtype=float)
HIGH_B = transform_amyloid_to_b(AD_AMYLOID)
B_BY_SEVERITY = {
    severity: BASELINE_B + severity * (HIGH_B - BASELINE_B)
    for severity in (0.0, 0.5, 1.0)
}

if not np.isfinite(HIGH_B).all():
    raise RuntimeError("The transformed inhibitory vector contains nonfinite values.")
if HIGH_B.min() < 0.0199 or HIGH_B.max() > 0.0701:
    raise RuntimeError("The transformed inhibitory vector is outside the expected range.")

pathology_summary_df = pd.DataFrame(
    [
        {
            "severity": severity,
            "condition": SEVERITY_LABELS[severity],
            "b_min": float(values.min()),
            "b_mean": float(values.mean()),
            "b_max": float(values.max()),
            "mean_tau_i_ms": float(np.mean(1.0 / values)),
        }
        for severity, values in B_BY_SEVERITY.items()
    ]
)
display(pathology_summary_df)

roi_pathology_df = roi_definition_df.copy()
roi_pathology_df["surrogate_amyloid"] = [
    AD_AMYLOID[LABEL_TO_INDEX[label]]
    for label in roi_pathology_df["label"]
]
roi_pathology_df["baseline_b"] = 0.07
roi_pathology_df["high_b"] = [
    HIGH_B[LABEL_TO_INDEX[label]] for label in roi_pathology_df["label"]
]
roi_pathology_df["b_reduction"] = (
    roi_pathology_df["baseline_b"] - roi_pathology_df["high_b"]
)
display(roi_pathology_df)
print(
    "Important: the target parcels have different local surrogate amyloid values. "
    "The local-dynamics-held-baseline counterfactual later in the notebook is therefore required."
)

## 8. TVB model and stimulus functions

Each simulation builds fresh TVB model and connectivity objects. The numerical arrays are passed explicitly into worker processes so joblib can memory-map the shared 379-by-379 structural matrix instead of serializing a separate copy for every task.

In [ ]:
def fresh_connectivity(weights, labels):
    weights = np.asarray(weights, dtype=float)
    labels = np.asarray(labels)
    if weights.shape != (N_REGIONS, N_REGIONS):
        raise ValueError("weights must be a 379-by-379 matrix.")
    if labels.shape != (N_REGIONS,):
        raise ValueError("labels must contain exactly 379 entries.")
    return connectivity.Connectivity(
        weights=weights.copy(),
        tract_lengths=np.zeros_like(weights),
        centres=np.zeros((N_REGIONS, 3), dtype=float),
        region_labels=labels.copy(),
        speed=np.array([100.0]),
    )

def build_model(b_values):
    background = np.array([0.1085])
    model = models.JansenRit(
        v0=np.array([6.0]),
        mu=background,
        p_min=background,
        p_max=background,
        b=np.asarray(b_values, dtype=float),
        variables_of_interest=("y1", "y2"),
    )
    # External input p(t) enters the y4 derivative in TVB's Jansen-Rit model.
    model.stvar = np.array([4], dtype=np.int32)
    return model

def make_temporal_equation(
    probe,
    model,
    input_peak_per_ms,
    onset_ms=STIMULUS_ONSET_MS,
    offset_ms=SIMULATION_MS,
):
    derivative_peak = float(
        model.A[0] * model.a[0] * float(input_peak_per_ms)
    )
    if probe == "pulse":
        return equations.TemporalApplicableEquation(
            equation=(
                "where((var >= onset) & (var < onset + width), amp, 0.0)"
            ),
            parameters={
                "onset": float(onset_ms),
                "width": float(PULSE_WIDTH_MS),
                "amp": derivative_peak,
            },
        )

    frequency_per_ms = {"2Hz": 0.002, "5Hz": 0.005}[probe]
    return equations.TemporalApplicableEquation(
        equation=(
            "where((var >= onset) & (var <= offset), "
            "0.5 * amp * (1.0 + sin(6.283185307179586 * "
            "frequency * (var - onset))), 0.0)"
        ),
        parameters={
            "onset": float(onset_ms),
            "offset": float(offset_ms),
            "amp": derivative_peak,
            "frequency": frequency_per_ms,
        },
    )

def make_initial_conditions(seed, dt_ms):
    rng = np.random.default_rng(int(seed))
    # Zero delays require only one state-history sample.
    return rng.random((1, 6, N_REGIONS, 1))

def run_tvb(
    b_values,
    probe,
    global_coupling,
    input_peak_per_ms,
    seed,
    weights,
    labels,
    a1_indices,
    dt_ms=MAIN_DT_MS,
    simulation_ms=SIMULATION_MS,
):
    b_values = np.asarray(b_values, dtype=float)
    if b_values.shape != (N_REGIONS,) or not np.isfinite(b_values).all():
        raise ValueError("b_values must be a finite 379-element vector.")
    if probe not in {None, "pulse", "2Hz", "5Hz"}:
        raise ValueError(f"Unknown probe: {probe}")
    if MONITOR_PERIOD_MS % float(dt_ms) != 0:
        raise ValueError("Monitor period must be an integer multiple of dt.")
    a1_indices = np.asarray(a1_indices, dtype=int)
    if a1_indices.ndim != 1 or len(a1_indices) == 0:
        raise ValueError("a1_indices must be a nonempty one-dimensional array.")

    white_matter = fresh_connectivity(weights, labels)
    model = build_model(b_values)
    stimulus = None
    if probe is not None:
        regional_weights = np.zeros(N_REGIONS, dtype=float)
        regional_weights[a1_indices] = 1.0 / np.sqrt(len(a1_indices))
        stimulus = patterns.StimuliRegion(
            temporal=make_temporal_equation(
                probe,
                model,
                input_peak_per_ms,
                offset_ms=simulation_ms,
            ),
            connectivity=white_matter,
            weight=regional_weights,
        )

    experiment = simulator.Simulator(
        connectivity=white_matter,
        model=model,
        coupling=coupling.SigmoidalJansenRit(
            a=np.array([float(global_coupling)])
        ),
        integrator=integrators.HeunDeterministic(dt=float(dt_ms)),
        monitors=(monitors.SubSample(period=MONITOR_PERIOD_MS),),
        stimulus=stimulus,
        initial_conditions=make_initial_conditions(seed, dt_ms),
    )
    experiment.configure()

    started = time.perf_counter()
    (time_ms, raw), = experiment.run(simulation_length=float(simulation_ms))
    wall_seconds = time.perf_counter() - started
    psp = raw[:, 0, :, 0] - raw[:, 1, :, 0]
    time_ms = np.asarray(time_ms, dtype=float)
    psp = np.asarray(psp, dtype=float)

    if psp.shape[1] != N_REGIONS:
        raise RuntimeError(f"Unexpected TVB output shape: {psp.shape}")
    if not np.isfinite(psp).all():
        raise RuntimeError("TVB produced NaN or infinite values.")
    if float(np.max(np.abs(psp))) > 100.0:
        raise RuntimeError(
            "TVB activity exceeded the prespecified safety bound of 100."
        )
    return time_ms, psp, wall_seconds

## 9. Response metrics and experiment runner

The primary response for 2 Hz and 5 Hz is the fitted harmonic amplitude at the exact drive frequency. A sine, cosine, intercept, and linear trend are fitted to the last 3 seconds of the control-subtracted response.

For each target network:

\[
\text{transfer} =
\frac{\text{mean target harmonic amplitude}}
     {\text{mean bilateral A1 harmonic amplitude}}
\]

Each network is then normalized to its own baseline using a log-ratio.

The primary contrast is:

\[
\Delta_{\text{music-speech}} =
\log_2(\text{music transfer}/\text{music baseline})
- \log_2(\text{speech transfer}/\text{speech baseline})
\]

The prespecified secondary contrast is:

\[
\Delta_{\text{semantic-episodic}} =
\log_2(\text{semantic-associated transfer}/\text{semantic-associated baseline})
- \log_2(\text{episodic-associated transfer}/\text{episodic-associated baseline})
\]

Positive values mean that the first proxy changed more favorably than the second within this model. Neither contrast demonstrates preserved musical memory.

The pulse response is a secondary RMS propagation diagnostic. Ordinary coherence and raw correlation are intentionally excluded because deterministic sinusoidal forcing can saturate them near 1.0.

The runner parallelizes one complete condition-seed block per worker. Each block computes one unstimulated control and reuses it for every requested probe, avoiding redundant controls while keeping jobs independent.

In [ ]:
NETWORK_LABELS = {
    # Primary comparison groups.
    "music": MUSIC_LABELS,
    "speech": SPEECH_LABELS,

    # Combined shared trunk for a single relay summary.
    "shared_auditory_relay": ordered_union(
        ROI_GROUPS["shared_early_auditory_relay"]["labels"],
        ROI_GROUPS["shared_parabelt"]["labels"],
        ROI_GROUPS["shared_auditory_association"]["labels"],
    ),
}

# Also save each anatomical stage and diagnostic subnetwork separately.
NETWORK_LABELS.update(
    {
        group_name: specification["labels"]
        for group_name, specification in ROI_GROUPS.items()
        if group_name != "a1_input"
    }
)

NETWORK_INDICES = {
    network_name: np.array(
        [LABEL_TO_INDEX[label] for label in network_labels],
        dtype=int,
    )
    for network_name, network_labels in NETWORK_LABELS.items()
}

KEY_NETWORKS = (
    "music",
    "speech",
    "music_semantic_task_associated",
    "music_episodic_task_associated",
    "shared_auditory_relay",
)

def harmonic_amplitude(time_ms, evoked, probe):
    frequency_hz = {"2Hz": 2.0, "5Hz": 5.0}[probe]
    window = (
        (time_ms >= PERIODIC_ANALYSIS_START_MS)
        & (time_ms <= SIMULATION_MS)
    )
    t_seconds = (time_ms[window] - PERIODIC_ANALYSIS_START_MS) / 1000.0
    y = np.asarray(evoked[window], dtype=float)
    if len(t_seconds) < 100:
        raise RuntimeError("Periodic analysis window is unexpectedly short.")

    omega_t = 2.0 * np.pi * frequency_hz * t_seconds
    design = np.column_stack(
        [
            np.sin(omega_t),
            np.cos(omega_t),
            np.ones_like(t_seconds),
            t_seconds - np.mean(t_seconds),
        ]
    )
    coefficients, _, _, _ = np.linalg.lstsq(design, y, rcond=None)
    fitted = design @ coefficients
    amplitude = np.sqrt(coefficients[0] ** 2 + coefficients[1] ** 2)

    residual_ss = np.sum((y - fitted) ** 2, axis=0)
    total_ss = np.sum((y - np.mean(y, axis=0)) ** 2, axis=0)
    r_squared = 1.0 - residual_ss / np.maximum(total_ss, 1e-15)
    return amplitude, r_squared

def pulse_rms(time_ms, evoked):
    window = (
        (time_ms >= STIMULUS_ONSET_MS)
        & (time_ms <= PULSE_ANALYSIS_END_MS)
    )
    y = np.asarray(evoked[window], dtype=float)
    if y.shape[0] < 100:
        raise RuntimeError("Pulse analysis window is unexpectedly short.")
    response = np.sqrt(np.mean(y ** 2, axis=0))
    return response, np.full(N_REGIONS, np.nan)

def extract_node_response(time_ms, evoked, probe):
    if probe == "pulse":
        return pulse_rms(time_ms, evoked)
    return harmonic_amplitude(time_ms, evoked, probe)

def b_signature(values):
    return hashlib.sha256(np.asarray(values, dtype=np.float64).tobytes()).hexdigest()[:16]

def _execute_condition_seed_block(
    job,
    weights,
    labels,
    a1_indices,
    network_index_items,
):
    """Execute one control plus all probes for one condition and seed."""
    ordinal = int(job["ordinal"])
    scope = str(job["scope"])
    severity = float(job["severity"])
    condition_name = str(job["condition"])
    variant = str(job["variant"])
    seed = int(job["seed"])
    probes = tuple(job["probes"])
    b_values = np.asarray(job["b_values"], dtype=float)
    global_coupling = float(job["global_coupling"])
    input_peak_per_ms = float(job["input_peak_per_ms"])
    dt_ms = float(job["dt_ms"])
    simulation_ms = float(job["simulation_ms"])
    worker_pid = int(os.getpid())

    node_rows = []
    network_rows = []
    manifest_rows = []

    control_time, control_psp, control_wall = run_tvb(
        b_values=b_values,
        probe=None,
        global_coupling=global_coupling,
        input_peak_per_ms=input_peak_per_ms,
        seed=seed,
        weights=weights,
        labels=labels,
        a1_indices=a1_indices,
        dt_ms=dt_ms,
        simulation_ms=simulation_ms,
    )
    manifest_rows.append(
        {
            "scope": scope,
            "variant": variant,
            "condition": condition_name,
            "severity": severity,
            "seed": seed,
            "probe": "none",
            "simulation_type": "matched_control",
            "global_coupling": global_coupling,
            "input_peak_per_ms": input_peak_per_ms,
            "dt_ms": dt_ms,
            "simulation_ms": simulation_ms,
            "b_signature": b_signature(b_values),
            "wall_seconds": float(control_wall),
            "max_abs_psp": float(np.max(np.abs(control_psp))),
            "max_abs_evoked": np.nan,
            "job_ordinal": ordinal,
            "worker_pid": worker_pid,
        }
    )

    for probe in probes:
        stimulated_time, stimulated_psp, stimulated_wall = run_tvb(
            b_values=b_values,
            probe=probe,
            global_coupling=global_coupling,
            input_peak_per_ms=input_peak_per_ms,
            seed=seed,
            weights=weights,
            labels=labels,
            a1_indices=a1_indices,
            dt_ms=dt_ms,
            simulation_ms=simulation_ms,
        )
        if not np.allclose(control_time, stimulated_time):
            raise RuntimeError("Control and stimulated time axes differ.")

        evoked = stimulated_psp - control_psp
        node_response, node_fit_r2 = extract_node_response(
            stimulated_time,
            evoked,
            probe,
        )
        if not np.isfinite(node_response).all():
            raise RuntimeError("A response metric is nonfinite.")

        a1_response = float(np.mean(node_response[a1_indices]))
        if a1_response <= 1e-8:
            raise RuntimeError(
                "A1 response is too small for stable normalization: "
                f"{a1_response}"
            )

        metric_name = (
            "pulse_rms" if probe == "pulse" else "harmonic_amplitude"
        )
        common = {
            "scope": scope,
            "variant": variant,
            "condition": condition_name,
            "severity": severity,
            "seed": seed,
            "probe": probe,
            "metric": metric_name,
            "global_coupling": global_coupling,
            "input_peak_per_ms": input_peak_per_ms,
            "dt_ms": dt_ms,
            "b_signature": b_signature(b_values),
        }

        for region_index in range(N_REGIONS):
            node_rows.append(
                {
                    **common,
                    "region_index": region_index,
                    "region_label": labels[region_index],
                    "response": float(node_response[region_index]),
                    "fit_r_squared": (
                        float(node_fit_r2[region_index])
                        if np.isfinite(node_fit_r2[region_index])
                        else np.nan
                    ),
                    "b_value": float(b_values[region_index]),
                }
            )

        for network, indices in network_index_items:
            indices = np.asarray(indices, dtype=int)
            network_response = float(np.mean(node_response[indices]))
            network_rows.append(
                {
                    **common,
                    "network": network,
                    "network_response": network_response,
                    "a1_response": a1_response,
                    "transfer": network_response / a1_response,
                    "median_target_fit_r_squared": (
                        float(np.nanmedian(node_fit_r2[indices]))
                        if probe != "pulse"
                        else np.nan
                    ),
                }
            )

        manifest_rows.append(
            {
                **common,
                "simulation_type": "stimulated",
                "simulation_ms": simulation_ms,
                "wall_seconds": float(stimulated_wall),
                "max_abs_psp": float(
                    np.max(np.abs(stimulated_psp))
                ),
                "max_abs_evoked": float(np.max(np.abs(evoked))),
                "job_ordinal": ordinal,
                "worker_pid": worker_pid,
            }
        )
        del stimulated_psp, evoked
        gc.collect()

    del control_psp
    gc.collect()
    return {
        "ordinal": ordinal,
        "node_rows": node_rows,
        "network_rows": network_rows,
        "manifest_rows": manifest_rows,
    }


def execute_grid(
    scope,
    conditions,
    seeds,
    probes,
    global_coupling,
    input_peak_per_ms,
    dt_ms=MAIN_DT_MS,
    simulation_ms=SIMULATION_MS,
):
    """Execute condition-seed blocks concurrently and aggregate by ordinal."""
    conditions = list(conditions)
    seeds = [int(seed) for seed in seeds]
    probes = tuple(probes)
    if not conditions or not seeds or not probes:
        raise ValueError("conditions, seeds, and probes must be nonempty.")

    jobs = []
    for condition in conditions:
        severity = float(condition["severity"])
        condition_name = str(condition["condition"])
        b_values = np.asarray(condition["b_values"], dtype=float)
        variant = str(condition.get("variant", condition_name))
        job_scope = str(condition.get("scope", scope))
        job_coupling = float(
            condition.get("global_coupling", global_coupling)
        )
        job_input_peak = float(
            condition.get("input_peak_per_ms", input_peak_per_ms)
        )

        for seed in seeds:
            jobs.append(
                {
                    "ordinal": len(jobs),
                    "scope": job_scope,
                    "variant": variant,
                    "condition": condition_name,
                    "severity": severity,
                    "b_values": b_values,
                    "seed": seed,
                    "probes": probes,
                    "global_coupling": job_coupling,
                    "input_peak_per_ms": job_input_peak,
                    "dt_ms": float(dt_ms),
                    "simulation_ms": float(simulation_ms),
                }
            )

    network_index_items = tuple(
        (
            network_name,
            np.asarray(indices, dtype=int),
        )
        for network_name, indices in NETWORK_INDICES.items()
    )
    outcomes = run_parallel_jobs(
        _execute_condition_seed_block,
        jobs,
        (
            WEIGHTS,
            LABELS,
            A1_INDICES,
            network_index_items,
        ),
        f"{scope} condition-seed blocks",
    )

    node_rows = []
    network_rows = []
    manifest_rows = []
    for outcome in sorted(outcomes, key=lambda item: item["ordinal"]):
        node_rows.extend(outcome["node_rows"])
        network_rows.extend(outcome["network_rows"])
        manifest_rows.extend(outcome["manifest_rows"])

    expected_node_rows = len(jobs) * len(probes) * N_REGIONS
    expected_network_rows = (
        len(jobs) * len(probes) * len(NETWORK_INDICES)
    )
    expected_manifest_rows = len(jobs) * (1 + len(probes))
    if len(node_rows) != expected_node_rows:
        raise RuntimeError(
            f"Expected {expected_node_rows} node rows, "
            f"received {len(node_rows)}."
        )
    if len(network_rows) != expected_network_rows:
        raise RuntimeError(
            f"Expected {expected_network_rows} network rows, "
            f"received {len(network_rows)}."
        )
    if len(manifest_rows) != expected_manifest_rows:
        raise RuntimeError(
            f"Expected {expected_manifest_rows} manifest rows, "
            f"received {len(manifest_rows)}."
        )

    return (
        pd.DataFrame(node_rows),
        pd.DataFrame(network_rows),
        pd.DataFrame(manifest_rows),
    )


def normalize_to_baseline(network_df, baseline_df=None):
    target = network_df.copy()
    source = target if baseline_df is None else baseline_df.copy()
    baseline = source[source["severity"] == 0.0][
        ["seed", "probe", "network", "transfer"]
    ].rename(columns={"transfer": "baseline_transfer"})
    target = target.merge(
        baseline,
        on=["seed", "probe", "network"],
        how="left",
        validate="many_to_one",
    )
    if target["baseline_transfer"].isna().any():
        raise RuntimeError("A baseline transfer value is missing.")
    target["log2_transfer_vs_baseline"] = np.log2(
        np.maximum(target["transfer"], 1e-15)
        / np.maximum(target["baseline_transfer"], 1e-15)
    )
    return target

def make_contrasts(normalized_df):
    pivot = normalized_df.pivot_table(
        index=[
            "scope",
            "variant",
            "condition",
            "severity",
            "seed",
            "probe",
            "global_coupling",
            "input_peak_per_ms",
            "dt_ms",
        ],
        columns="network",
        values="log2_transfer_vs_baseline",
    ).reset_index()
    required_primary = {"music", "speech"}
    required_secondary = {
        "music_semantic_task_associated",
        "music_episodic_task_associated",
    }
    if not required_primary.issubset(pivot.columns):
        raise RuntimeError("Both primary network results are required.")
    if not required_secondary.issubset(pivot.columns):
        raise RuntimeError("Both musical-memory proxy results are required.")
    pivot["music_minus_speech_log2_change"] = (
        pivot["music"] - pivot["speech"]
    )
    pivot["semantic_minus_episodic_log2_change"] = (
        pivot["music_semantic_task_associated"]
        - pivot["music_episodic_task_associated"]
    )
    return pivot

## 10. Baseline-only coupling calibration

Each candidate coupling is independent, so the calibration scan is distributed across the same CPU worker pool. Every worker runs the matched control and pulse pair for one candidate coupling.

In [ ]:
def _run_calibration_block(
    job,
    weights,
    labels,
    a1_indices,
    music_indices,
    speech_indices,
):
    candidate_g = float(job["global_coupling"])
    calibration_seed = int(job["seed"])
    calibration_simulation_ms = float(job["simulation_ms"])
    ordinal = int(job["ordinal"])

    t_control, y_control, control_wall = run_tvb(
        b_values=BASELINE_B,
        probe=None,
        global_coupling=candidate_g,
        input_peak_per_ms=MAIN_INPUT_PEAK_PER_MS,
        seed=calibration_seed,
        weights=weights,
        labels=labels,
        a1_indices=a1_indices,
        simulation_ms=calibration_simulation_ms,
    )
    t_pulse, y_pulse, pulse_wall = run_tvb(
        b_values=BASELINE_B,
        probe="pulse",
        global_coupling=candidate_g,
        input_peak_per_ms=MAIN_INPUT_PEAK_PER_MS,
        seed=calibration_seed,
        weights=weights,
        labels=labels,
        a1_indices=a1_indices,
        simulation_ms=calibration_simulation_ms,
    )
    if not np.allclose(t_control, t_pulse):
        raise RuntimeError("Calibration time axes differ.")

    evoked = y_pulse - y_control
    response, _ = pulse_rms(t_pulse, evoked)
    a1 = float(np.mean(response[a1_indices]))
    music = float(np.mean(response[music_indices]))
    speech = float(np.mean(response[speech_indices]))
    return {
        "ordinal": ordinal,
        "row": {
            "global_coupling": candidate_g,
            "a1_rms": a1,
            "music_transfer": music / a1,
            "speech_transfer": speech / a1,
            "balanced_target_score": math.sqrt(
                (music / a1) * (speech / a1)
            ),
            "max_abs_evoked": float(np.max(np.abs(evoked))),
            "wall_seconds": float(control_wall + pulse_wall),
            "worker_pid": int(os.getpid()),
        },
    }


calibration_simulation_ms = 4000.0
calibration_seed = CFG["seeds"][0]
calibration_jobs = [
    {
        "ordinal": ordinal,
        "global_coupling": candidate_g,
        "seed": calibration_seed,
        "simulation_ms": calibration_simulation_ms,
    }
    for ordinal, candidate_g in enumerate(
        CFG["calibration_couplings"]
    )
]
calibration_outcomes = run_parallel_jobs(
    _run_calibration_block,
    calibration_jobs,
    (
        WEIGHTS,
        LABELS,
        A1_INDICES,
        MUSIC_INDICES,
        SPEECH_INDICES,
    ),
    "baseline coupling candidates",
)
calibration_rows = [
    outcome["row"]
    for outcome in sorted(
        calibration_outcomes,
        key=lambda item: item["ordinal"],
    )
]

calibration_df = pd.DataFrame(calibration_rows)
display(calibration_df)
if not np.isfinite(calibration_df.select_dtypes("number")).all().all():
    raise RuntimeError("Calibration produced a nonfinite value.")
selected_row = calibration_df[
    calibration_df["global_coupling"] == MAIN_GLOBAL_COUPLING
]
if selected_row.empty:
    warnings.warn(
        "The selected coupling was not included in this shortened scan."
    )
elif float(selected_row["max_abs_evoked"].iloc[0]) >= 50.0:
    raise RuntimeError("The selected coupling produced a saturated response.")

fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.plot(
    calibration_df["global_coupling"],
    calibration_df["music_transfer"],
    marker="o",
    label="Music proxy",
)
ax.plot(
    calibration_df["global_coupling"],
    calibration_df["speech_transfer"],
    marker="o",
    label="Speech proxy",
)
ax.axvline(
    MAIN_GLOBAL_COUPLING,
    color="black",
    linestyle="--",
    label="Main G",
)
ax.set(
    xlabel="Global coupling G",
    ylabel="Pulse target RMS / A1 RMS",
    title="Baseline-only coupling calibration",
)
ax.legend()
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "01_baseline_coupling_calibration.png",
    dpi=180,
)
plt.show()

## 11. Run the main full-field experiment

Each stimulated run is paired with an unstimulated run that has the same inhibitory vector and initial condition. Both proxy networks are measured from every simulation, so the stimulus is never changed between the network comparisons.

In [ ]:
main_conditions = [
    {
        "condition": SEVERITY_LABELS[severity],
        "severity": severity,
        "b_values": B_BY_SEVERITY[severity],
        "variant": "full_field",
    }
    for severity in CFG["severities"]
]

main_node_df, main_network_df, main_manifest_df = execute_grid(
    scope="main_full_field",
    conditions=main_conditions,
    seeds=CFG["seeds"],
    probes=PROBES,
    global_coupling=MAIN_GLOBAL_COUPLING,
    input_peak_per_ms=MAIN_INPUT_PEAK_PER_MS,
)
main_normalized_df = normalize_to_baseline(main_network_df)
main_contrast_df = make_contrasts(main_normalized_df)

display(main_network_df.head())
display(
    main_contrast_df[
        main_contrast_df["probe"].isin(PERIODIC_PROBES)
    ].sort_values(["probe", "seed", "severity"])
)

In [ ]:
main_stage_summary_df = (
    main_normalized_df.groupby(
        ["probe", "severity", "condition", "network"], as_index=False
    )
    .agg(
        median_log2_change=("log2_transfer_vs_baseline", "median"),
        minimum_log2_change=("log2_transfer_vs_baseline", "min"),
        maximum_log2_change=("log2_transfer_vs_baseline", "max"),
        numerical_seeds=("seed", "nunique"),
    )
    .sort_values(["probe", "severity", "network"])
)

endpoint_mask = (
    (main_contrast_df["severity"] == 1.0)
    & (main_contrast_df["probe"].isin(PERIODIC_PROBES))
)
primary_endpoint_df = main_contrast_df[endpoint_mask].copy()
secondary_endpoint_df = main_contrast_df[endpoint_mask].copy()

secondary_main_contrast_df = main_contrast_df[
    [
        "scope",
        "variant",
        "condition",
        "severity",
        "seed",
        "probe",
        "global_coupling",
        "input_peak_per_ms",
        "dt_ms",
        "music_semantic_task_associated",
        "music_episodic_task_associated",
        "semantic_minus_episodic_log2_change",
    ]
].copy()

display(
    main_stage_summary_df[
        main_stage_summary_df["network"].isin(KEY_NETWORKS)
    ]
)
display(
    primary_endpoint_df[
        [
            "seed",
            "probe",
            "music",
            "speech",
            "music_minus_speech_log2_change",
        ]
    ]
)
display(
    secondary_endpoint_df[
        [
            "seed",
            "probe",
            "music_semantic_task_associated",
            "music_episodic_task_associated",
            "semantic_minus_episodic_log2_change",
        ]
    ]
)
print(
    "These ranges summarize numerical initial-condition sensitivity. "
    "They are not participant confidence intervals."
)

## 12. Required local-dynamics-held-baseline counterfactuals

Regional surrogate amyloid values differ across target parcels. A contrast can therefore arise from different local \(b\) changes rather than different network propagation.

Two separate counterfactual endpoint simulations are required:

1. **Primary counterfactual:** hold A1 plus the primary music and speech targets at baseline \(b\), while leaving shared relays and all other parcels perturbed.
2. **Secondary counterfactual:** hold A1 plus the semantic- and episodic-task-associated targets at baseline \(b\), while leaving all other parcels perturbed.

The masks are kept separate so that freezing the secondary targets cannot silently change the primary control, and vice versa.

In [ ]:
PRIMARY_LOCAL_FIXED_HIGH_B = HIGH_B.copy()
PRIMARY_LOCAL_FIXED_HIGH_B[PRIMARY_COUNTERFACTUAL_FIXED_INDICES] = (
    BASELINE_B[PRIMARY_COUNTERFACTUAL_FIXED_INDICES]
)

MEMORY_LOCAL_FIXED_HIGH_B = HIGH_B.copy()
MEMORY_LOCAL_FIXED_HIGH_B[MEMORY_COUNTERFACTUAL_FIXED_INDICES] = (
    BASELINE_B[MEMORY_COUNTERFACTUAL_FIXED_INDICES]
)

local_fixed_conditions = [
    {
        "condition": "High AD-like perturbation, primary local dynamics fixed",
        "severity": 1.0,
        "b_values": PRIMARY_LOCAL_FIXED_HIGH_B,
        "variant": "primary_local_fixed_endpoint",
    },
    {
        "condition": (
            "High AD-like perturbation, musical-memory proxy "
            "local dynamics fixed"
        ),
        "severity": 1.0,
        "b_values": MEMORY_LOCAL_FIXED_HIGH_B,
        "variant": "memory_local_fixed_endpoint",
    },
]
(
    local_fixed_node_df,
    local_fixed_network_df,
    local_fixed_manifest_df,
) = execute_grid(
    scope="local_dynamics_counterfactual",
    conditions=local_fixed_conditions,
    seeds=CFG["seeds"],
    probes=PROBES,
    global_coupling=MAIN_GLOBAL_COUPLING,
    input_peak_per_ms=MAIN_INPUT_PEAK_PER_MS,
)
local_fixed_normalized_df = normalize_to_baseline(
    local_fixed_network_df,
    baseline_df=main_network_df,
)
local_fixed_contrast_df = make_contrasts(local_fixed_normalized_df)

primary_local_fixed_contrast_df = local_fixed_contrast_df[
    local_fixed_contrast_df["variant"]
    == "primary_local_fixed_endpoint"
].copy()
memory_local_fixed_contrast_df = local_fixed_contrast_df[
    local_fixed_contrast_df["variant"]
    == "memory_local_fixed_endpoint"
].copy()

counterfactual_comparison_df = pd.concat(
    [
        primary_endpoint_df.assign(analysis="Full regional perturbation"),
        primary_local_fixed_contrast_df[
            primary_local_fixed_contrast_df["probe"].isin(PERIODIC_PROBES)
        ].assign(analysis="A1 and primary targets locally fixed"),
    ],
    ignore_index=True,
)

memory_counterfactual_comparison_df = pd.concat(
    [
        secondary_endpoint_df.assign(analysis="Full regional perturbation"),
        memory_local_fixed_contrast_df[
            memory_local_fixed_contrast_df["probe"].isin(PERIODIC_PROBES)
        ].assign(analysis="A1 and memory-proxy targets locally fixed"),
    ],
    ignore_index=True,
)

display(
    counterfactual_comparison_df[
        [
            "analysis",
            "seed",
            "probe",
            "music",
            "speech",
            "music_minus_speech_log2_change",
        ]
    ].sort_values(["probe", "seed", "analysis"])
)
display(
    memory_counterfactual_comparison_df[
        [
            "analysis",
            "seed",
            "probe",
            "music_semantic_task_associated",
            "music_episodic_task_associated",
            "semantic_minus_episodic_log2_change",
        ]
    ].sort_values(["probe", "seed", "analysis"])
)

## 13. Topology- and pathology-matched control subnetworks

Both the primary and secondary contrasts are compared with size-preserving random parcel sets. Every declared target parcel is matched to a same-hemisphere cortical parcel using:

- log weighted structural strength;
- log direct A1 structural affinity;
- local high-endpoint \(b\) reduction.

The primary null pairs groups of 8 and 14 parcels. The secondary null pairs groups of 5 and 4 parcels. Declared pathway parcels are excluded from all control sets. These are simulation-level placement controls, not participant-level statistical tests.

In [ ]:
MATCH_WEIGHTS = 0.5 * (WEIGHTS + WEIGHTS.T)
CORTICAL_INDICES = np.arange(360)
weighted_strength = MATCH_WEIGHTS.sum(axis=1)
direct_a1_affinity = MATCH_WEIGHTS[:, A1_INDICES].sum(axis=1)
local_b_reduction = BASELINE_B - HIGH_B

raw_features = np.column_stack(
    [
        np.log10(weighted_strength + 1e-15),
        np.log10(direct_a1_affinity + 1e-15),
        local_b_reduction,
    ]
)
cortical_mean = raw_features[CORTICAL_INDICES].mean(axis=0)
cortical_std = raw_features[CORTICAL_INDICES].std(axis=0)
if np.any(cortical_std <= 0):
    raise RuntimeError("A matching feature has zero variance.")
matching_z = (raw_features - cortical_mean) / cortical_std

target_groups_for_matching = {
    "music": MUSIC_INDICES,
    "speech": SPEECH_INDICES,
    "semantic_associated": SEMANTIC_MEMORY_INDICES,
    "episodic_associated": EPISODIC_MEMORY_INDICES,
}
target_feature_rows = []
for group_name, indices in target_groups_for_matching.items():
    for index in indices:
        target_feature_rows.append(
            {
                "label": LABELS[index],
                "network": group_name,
                "weighted_strength": weighted_strength[index],
                "direct_A1_affinity": direct_a1_affinity[index],
                "local_b_reduction": local_b_reduction[index],
            }
        )
target_feature_df = pd.DataFrame(target_feature_rows)
display(target_feature_df)


def same_hemisphere_candidates(target_index):
    if target_index < 180:
        return np.arange(0, 180)
    if target_index < 360:
        return np.arange(180, 360)
    raise ValueError("Matched targets must be cortical.")


def draw_matched_set(target_indices, rng, reserved, top_k=30):
    selected = []
    distances_selected = []
    for target_index in target_indices:
        candidates = same_hemisphere_candidates(int(target_index))
        candidates = np.array(
            [
                index
                for index in candidates
                if index not in reserved and index not in selected
            ],
            dtype=int,
        )
        distances = np.linalg.norm(
            matching_z[candidates] - matching_z[target_index], axis=1
        )
        order = np.argsort(distances)
        pool_order = order[: min(top_k, len(order))]
        pool = candidates[pool_order]
        pool_distances = distances[pool_order]
        scale = max(float(np.median(pool_distances)), 1e-6)
        probabilities = np.exp(
            -(pool_distances - pool_distances.min()) / scale
        )
        probabilities /= probabilities.sum()
        chosen = int(rng.choice(pool, p=probabilities))
        selected.append(chosen)
        distances_selected.append(
            float(
                np.linalg.norm(
                    matching_z[chosen] - matching_z[target_index]
                )
            )
        )
    return np.array(selected, dtype=int), distances_selected


def build_matched_pair_sets(
    pair_name,
    left_name,
    left_indices,
    right_name,
    right_indices,
    rng_seed,
):
    rng = np.random.default_rng(rng_seed)
    excluded = set(ALL_DECLARED_INDICES.tolist())
    rows = []
    for set_id in range(CFG["matched_null_sets"]):
        left_control, left_distances = draw_matched_set(
            left_indices,
            rng,
            reserved=excluded,
        )
        right_control, right_distances = draw_matched_set(
            right_indices,
            rng,
            reserved=excluded.union(left_control.tolist()),
        )
        rows.append(
            {
                "pair_name": pair_name,
                "set_id": set_id,
                "left_name": left_name,
                "right_name": right_name,
                "left_control_indices": ";".join(map(str, left_control)),
                "right_control_indices": ";".join(map(str, right_control)),
                "left_control_labels": ";".join(LABELS[left_control]),
                "right_control_labels": ";".join(LABELS[right_control]),
                "mean_standardized_match_distance": float(
                    np.mean(left_distances + right_distances)
                ),
            }
        )
    return pd.DataFrame(rows)


matched_sets_df = build_matched_pair_sets(
    pair_name="music_minus_speech",
    left_name="music",
    left_indices=MUSIC_INDICES,
    right_name="speech",
    right_indices=SPEECH_INDICES,
    rng_seed=20260727,
)
memory_matched_sets_df = build_matched_pair_sets(
    pair_name="semantic_minus_episodic",
    left_name="semantic_associated",
    left_indices=SEMANTIC_MEMORY_INDICES,
    right_name="episodic_associated",
    right_indices=EPISODIC_MEMORY_INDICES,
    rng_seed=20260728,
)

display(matched_sets_df.head())
display(matched_sets_df["mean_standardized_match_distance"].describe())
display(memory_matched_sets_df.head())
display(
    memory_matched_sets_df[
        "mean_standardized_match_distance"
    ].describe()
)

In [ ]:
def node_response_vector(node_df, seed, probe, severity):
    subset = node_df[
        (node_df["seed"] == seed)
        & (node_df["probe"] == probe)
        & (node_df["severity"] == severity)
    ].sort_values("region_index")
    if len(subset) != N_REGIONS:
        raise RuntimeError("Expected exactly one response per region.")
    return subset["response"].to_numpy()


def a1_response_value(network_df, seed, probe, severity):
    subset = network_df[
        (network_df["seed"] == seed)
        & (network_df["probe"] == probe)
        & (network_df["severity"] == severity)
    ]
    values = subset["a1_response"].unique()
    if len(values) != 1:
        raise RuntimeError("Expected one A1 response value.")
    return float(values[0])


def compute_matched_null(pair_sets_df):
    rows = []
    for seed in CFG["seeds"]:
        for probe in PERIODIC_PROBES:
            baseline_response = node_response_vector(
                main_node_df, seed, probe, 0.0
            )
            high_response = node_response_vector(
                main_node_df, seed, probe, 1.0
            )
            baseline_a1 = a1_response_value(
                main_network_df, seed, probe, 0.0
            )
            high_a1 = a1_response_value(
                main_network_df, seed, probe, 1.0
            )

            for row in pair_sets_df.itertuples(index=False):
                left_control = np.fromstring(
                    row.left_control_indices, sep=";", dtype=int
                )
                right_control = np.fromstring(
                    row.right_control_indices, sep=";", dtype=int
                )
                left_baseline_transfer = (
                    np.mean(baseline_response[left_control]) / baseline_a1
                )
                left_high_transfer = (
                    np.mean(high_response[left_control]) / high_a1
                )
                right_baseline_transfer = (
                    np.mean(baseline_response[right_control]) / baseline_a1
                )
                right_high_transfer = (
                    np.mean(high_response[right_control]) / high_a1
                )
                left_change = np.log2(
                    max(left_high_transfer, 1e-15)
                    / max(left_baseline_transfer, 1e-15)
                )
                right_change = np.log2(
                    max(right_high_transfer, 1e-15)
                    / max(right_baseline_transfer, 1e-15)
                )
                rows.append(
                    {
                        "pair_name": row.pair_name,
                        "set_id": row.set_id,
                        "seed": seed,
                        "probe": probe,
                        "left_name": row.left_name,
                        "right_name": row.right_name,
                        "left_control_log2_change": left_change,
                        "right_control_log2_change": right_change,
                        "null_left_minus_right": (
                            left_change - right_change
                        ),
                        "mean_standardized_match_distance": (
                            row.mean_standardized_match_distance
                        ),
                    }
                )
    return pd.DataFrame(rows)


def summarize_matched_null(
    observed_df,
    observed_column,
    null_df,
    null_column,
):
    rows = []
    for observed in observed_df.itertuples(index=False):
        null_values = null_df[
            (null_df["seed"] == observed.seed)
            & (null_df["probe"] == observed.probe)
        ][null_column].to_numpy()
        observed_value = getattr(observed, observed_column)
        rows.append(
            {
                "seed": observed.seed,
                "probe": observed.probe,
                "observed_contrast": observed_value,
                "null_median": float(np.median(null_values)),
                "null_5th_percentile": float(
                    np.quantile(null_values, 0.05)
                ),
                "null_95th_percentile": float(
                    np.quantile(null_values, 0.95)
                ),
                "observed_percentile_within_simulation_null": float(
                    100.0 * np.mean(null_values <= observed_value)
                ),
            }
        )
    return pd.DataFrame(rows)


matched_null_df = compute_matched_null(matched_sets_df).rename(
    columns={
        "left_control_log2_change": "music_control_log2_change",
        "right_control_log2_change": "speech_control_log2_change",
        "null_left_minus_right": "null_music_minus_speech",
    }
)
memory_matched_null_df = compute_matched_null(
    memory_matched_sets_df
).rename(
    columns={
        "left_control_log2_change": (
            "semantic_control_log2_change"
        ),
        "right_control_log2_change": (
            "episodic_control_log2_change"
        ),
        "null_left_minus_right": "null_semantic_minus_episodic",
    }
)

matched_null_summary_df = summarize_matched_null(
    primary_endpoint_df,
    "music_minus_speech_log2_change",
    matched_null_df,
    "null_music_minus_speech",
)
memory_matched_null_summary_df = summarize_matched_null(
    secondary_endpoint_df,
    "semantic_minus_episodic_log2_change",
    memory_matched_null_df,
    "null_semantic_minus_episodic",
)

display(matched_null_summary_df)
display(memory_matched_null_summary_df)

## 14. Coupling and stimulus-strength sensitivity

The full-field baseline and high endpoint are repeated at alternative global-coupling and input-strength values. Both the primary music-minus-speech contrast and the secondary semantic-minus-episodic contrast are computed from the same simulations. These checks test parameter dependence; they do not select a parameter value based on a preferred result.

All scenario-condition blocks are submitted together so the worker pool can use more than two cores.

In [ ]:
sensitivity_network_frames = []
sensitivity_manifest_frames = []

main_seed = CFG["seeds"][0]
main_sensitivity_source = main_network_df[
    (main_network_df["seed"] == main_seed)
    & (main_network_df["severity"].isin([0.0, 1.0]))
    & (main_network_df["probe"].isin(PERIODIC_PROBES))
].copy()
main_sensitivity_source["scope"] = "sensitivity_main"
main_sensitivity_source["variant"] = "G60_input_0.02"
sensitivity_network_frames.append(main_sensitivity_source)

sensitivity_conditions = []
for scenario in CFG["sensitivity_scenarios"]:
    scenario_name = str(scenario["scenario"])
    for severity in (0.0, 1.0):
        sensitivity_conditions.append(
            {
                "scope": f"sensitivity_{scenario_name}",
                "condition": SEVERITY_LABELS[severity],
                "severity": severity,
                "b_values": B_BY_SEVERITY[severity],
                "variant": scenario_name,
                "global_coupling": float(
                    scenario["global_coupling"]
                ),
                "input_peak_per_ms": float(
                    scenario["input_peak"]
                ),
            }
        )

(
    _,
    sensitivity_scenario_network_df,
    sensitivity_scenario_manifest_df,
) = execute_grid(
    scope="sensitivity_alternatives",
    conditions=sensitivity_conditions,
    seeds=[main_seed],
    probes=PERIODIC_PROBES,
    global_coupling=MAIN_GLOBAL_COUPLING,
    input_peak_per_ms=MAIN_INPUT_PEAK_PER_MS,
)
sensitivity_network_frames.append(
    sensitivity_scenario_network_df
)
sensitivity_manifest_frames.append(
    sensitivity_scenario_manifest_df
)

sensitivity_network_df = pd.concat(
    sensitivity_network_frames,
    ignore_index=True,
)
sensitivity_normalized_frames = []
for scenario_scope, group in sensitivity_network_df.groupby(
    "scope",
    sort=False,
):
    sensitivity_normalized_frames.append(
        normalize_to_baseline(group)
    )
sensitivity_normalized_df = pd.concat(
    sensitivity_normalized_frames,
    ignore_index=True,
)
sensitivity_contrast_df = make_contrasts(
    sensitivity_normalized_df
)
sensitivity_endpoint_df = sensitivity_contrast_df[
    sensitivity_contrast_df["severity"] == 1.0
].copy()
display(
    sensitivity_endpoint_df[
        [
            "variant",
            "probe",
            "global_coupling",
            "input_peak_per_ms",
            "music_minus_speech_log2_change",
            "semantic_minus_episodic_log2_change",
        ]
    ].sort_values(
        ["probe", "global_coupling", "input_peak_per_ms"]
    )
)

## 15. Spatial-placement sensitivity

The high-endpoint surrogate perturbation values are shuffled within the left-cortical, right-cortical, and subcortical blocks while preserving their distributions. The first numerical seed is used. Both primary and secondary contrasts are recomputed from each shuffled simulation.

This asks whether either observed contrast depends on the specific regional placement of the artificial perturbation. It does not create alternative patients. All shuffled conditions are generated deterministically in the parent process and then submitted in one parallel grid, which is essential for distributing the 100 final-mode shuffles across all allocated cores.

In [ ]:
shuffle_rng = np.random.default_rng(3792026)
shuffle_blocks = [
    np.arange(0, 180),
    np.arange(180, 360),
    np.arange(360, 379),
]
shuffle_conditions = []

for shuffle_id in range(CFG["spatial_shuffles"]):
    shuffled_b = HIGH_B.copy()
    for block in shuffle_blocks:
        shuffled_b[block] = shuffle_rng.permutation(
            HIGH_B[block]
        )
    shuffle_label = f"shuffle_{shuffle_id + 1:02d}"
    shuffle_conditions.append(
        {
            "scope": f"spatial_{shuffle_label}",
            "condition": (
                "High AD-like perturbation, spatially shuffled"
            ),
            "severity": 1.0,
            "b_values": shuffled_b,
            "variant": shuffle_label,
        }
    )

(
    _,
    shuffle_network_df,
    shuffle_manifest_df,
) = execute_grid(
    scope="spatial_shuffles",
    conditions=shuffle_conditions,
    seeds=[main_seed],
    probes=PERIODIC_PROBES,
    global_coupling=MAIN_GLOBAL_COUPLING,
    input_peak_per_ms=MAIN_INPUT_PEAK_PER_MS,
)
shuffle_network_frames = [shuffle_network_df]
shuffle_manifest_frames = [shuffle_manifest_df]

shuffle_normalized_frames = []
for shuffle_scope, group in shuffle_network_df.groupby(
    "scope",
    sort=False,
):
    shuffle_normalized_frames.append(
        normalize_to_baseline(
            group,
            baseline_df=main_network_df,
        )
    )
shuffle_normalized_df = pd.concat(
    shuffle_normalized_frames,
    ignore_index=True,
)
shuffle_contrast_df = make_contrasts(shuffle_normalized_df)

observed_first_seed_df = primary_endpoint_df[
    primary_endpoint_df["seed"] == main_seed
][["probe", "music_minus_speech_log2_change"]].rename(
    columns={
        "music_minus_speech_log2_change": "observed_contrast"
    }
)
shuffle_summary_df = (
    shuffle_contrast_df.groupby("probe", as_index=False)
    .agg(
        shuffle_median=(
            "music_minus_speech_log2_change",
            "median",
        ),
        shuffle_minimum=(
            "music_minus_speech_log2_change",
            "min",
        ),
        shuffle_maximum=(
            "music_minus_speech_log2_change",
            "max",
        ),
        spatial_shuffles=("variant", "nunique"),
    )
    .merge(
        observed_first_seed_df,
        on="probe",
        how="left",
    )
)
memory_observed_first_seed_df = secondary_endpoint_df[
    secondary_endpoint_df["seed"] == main_seed
][["probe", "semantic_minus_episodic_log2_change"]].rename(
    columns={
        "semantic_minus_episodic_log2_change": "observed_contrast"
    }
)
memory_shuffle_summary_df = (
    shuffle_contrast_df.groupby("probe", as_index=False)
    .agg(
        shuffle_median=(
            "semantic_minus_episodic_log2_change",
            "median",
        ),
        shuffle_minimum=(
            "semantic_minus_episodic_log2_change",
            "min",
        ),
        shuffle_maximum=(
            "semantic_minus_episodic_log2_change",
            "max",
        ),
        spatial_shuffles=("variant", "nunique"),
    )
    .merge(
        memory_observed_first_seed_df,
        on="probe",
        how="left",
    )
)
display(shuffle_summary_df)
display(memory_shuffle_summary_df)

## 16. Required integration-step convergence check

The baseline and high AD-like endpoints at both 2 Hz and 5 Hz are compared between the 0.5 ms main integration step and a 0.25 ms reference step. A targeted pre-final test showed that the earlier 1.0 ms step failed the 5% rule for the expanded primary speech proxy at high perturbation and 5 Hz, whereas 0.5 ms passed against 0.25 ms.

All saved networks are reported. The notebook stops if transfer differs by 5% or more for any inferential network in `KEY_NETWORKS`: the primary music and speech proxies, the two secondary musical-memory-task-associated proxies, and the shared auditory relay. Other context groups are descriptive and receive an explicit convergence flag but do not determine whether the primary or secondary tests may proceed.

Median target harmonic-fit \(R^2\) is reported separately. Low \(R^2\) means that a probe-locked sinusoid explains little of the target time-series variance; it is a signal-quality warning and must temper interpretation even when the exact-frequency transfer estimate is numerically converged.

In [ ]:
# Check both endpoints at both periodic probe frequencies.
DT_CHECK_SEVERITIES = (0.0, 1.0)
DT_CHECK_PROBES = PERIODIC_PROBES  # ("2Hz", "5Hz")
DT_CHECK_NETWORKS = KEY_NETWORKS
DT_RELATIVE_TOLERANCE = 0.05

_, dt_reference_network_df, dt_reference_manifest_df = execute_grid(
    scope="dt_reference_0.25ms",
    conditions=[
        {
            "condition": SEVERITY_LABELS[severity],
            "severity": severity,
            "b_values": B_BY_SEVERITY[severity],
            "variant": f"dt_0.25ms_severity_{severity:.1f}",
        }
        for severity in DT_CHECK_SEVERITIES
    ],
    seeds=[main_seed],
    probes=DT_CHECK_PROBES,
    global_coupling=MAIN_GLOBAL_COUPLING,
    input_peak_per_ms=MAIN_INPUT_PEAK_PER_MS,
    dt_ms=REFERENCE_DT_MS,
)

# Corresponding results from the 0.5 ms main experiment.
dt_main = main_network_df[
    (main_network_df["seed"] == main_seed)
    & (main_network_df["severity"].isin(DT_CHECK_SEVERITIES))
    & (main_network_df["probe"].isin(DT_CHECK_PROBES))
][
    [
        "severity",
        "probe",
        "network",
        "transfer",
        "median_target_fit_r_squared",
    ]
].rename(
    columns={
        "transfer": "transfer_dt_0.5ms",
        "median_target_fit_r_squared":
            "median_target_fit_r_squared_dt_0.5ms",
    }
)

# Results recomputed using the 0.25 ms reference step.
dt_reference = dt_reference_network_df[
    [
        "severity",
        "probe",
        "network",
        "transfer",
        "median_target_fit_r_squared",
    ]
].rename(
    columns={
        "transfer": "transfer_dt_0.25ms",
        "median_target_fit_r_squared":
            "median_target_fit_r_squared_dt_0.25ms",
    }
)

dt_convergence_df = dt_main.merge(
    dt_reference,
    on=["severity", "probe", "network"],
    validate="one_to_one",
)
dt_convergence_df["condition"] = dt_convergence_df[
    "severity"
].map(SEVERITY_LABELS)
dt_convergence_df["required_for_inference"] = (
    dt_convergence_df["network"].isin(DT_CHECK_NETWORKS)
)

dt_convergence_df["relative_difference"] = (
    (
        dt_convergence_df["transfer_dt_0.5ms"]
        - dt_convergence_df["transfer_dt_0.25ms"]
    ).abs()
    / dt_convergence_df["transfer_dt_0.25ms"].abs().clip(lower=1e-15)
)
dt_convergence_df["convergence_passed"] = (
    dt_convergence_df["relative_difference"]
    < DT_RELATIVE_TOLERANCE
)

# Report R² at both integration steps as a signal-quality diagnostic.
dt_convergence_df["fit_r_squared_difference"] = (
    dt_convergence_df["median_target_fit_r_squared_dt_0.5ms"]
    - dt_convergence_df["median_target_fit_r_squared_dt_0.25ms"]
).abs()

dt_convergence_df = dt_convergence_df.sort_values(
    ["required_for_inference", "severity", "probe", "network"],
    ascending=[False, True, True, True],
).reset_index(drop=True)

display(
    dt_convergence_df[
        [
            "condition",
            "probe",
            "network",
            "required_for_inference",
            "transfer_dt_0.5ms",
            "transfer_dt_0.25ms",
            "relative_difference",
            "convergence_passed",
            "median_target_fit_r_squared_dt_0.5ms",
            "median_target_fit_r_squared_dt_0.25ms",
            "fit_r_squared_difference",
        ]
    ]
)

failed_required_dt_rows = dt_convergence_df[
    dt_convergence_df["required_for_inference"]
    & ~dt_convergence_df["convergence_passed"]
]
if not failed_required_dt_rows.empty:
    display(failed_required_dt_rows)
    raise RuntimeError(
        "The 0.5 ms integration step failed the prespecified "
        "5% convergence check for at least one inferential network."
    )

nonconverged_descriptive_dt_rows = dt_convergence_df[
    ~dt_convergence_df["required_for_inference"]
    & ~dt_convergence_df["convergence_passed"]
]
if not nonconverged_descriptive_dt_rows.empty:
    print(
        "Warning: some descriptive context networks did not meet the "
        "5% rule. They remain saved with convergence_passed=False and "
        "must not be interpreted inferentially."
    )
    display(nonconverged_descriptive_dt_rows)

print(
    "Integration-step convergence check passed for every inferential "
    "network at baseline and high AD-like perturbation, at 2 Hz and 5 Hz."
)

## 17. Figures

Every numerical seed is shown when more than one is available. Thick lines summarize seed medians. These graphics describe model robustness and do not show human-subject uncertainty.

The primary figures remain music versus speech. Separate secondary figures show the semantic-task-associated versus episodic-task-associated proxy analysis so the secondary result cannot be mistaken for the primary endpoint.

In [ ]:
NETWORK_COLORS = {"music": "#2B6CB0", "speech": "#D97706"}
NETWORK_NAMES = {"music": "Music proxy", "speech": "Speech proxy"}

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), sharey=True)
for ax, probe in zip(axes, PERIODIC_PROBES):
    plot_data = main_normalized_df[main_normalized_df["probe"] == probe]
    for network in ("music", "speech"):
        network_data = plot_data[plot_data["network"] == network]
        for _, seed_data in network_data.groupby("seed"):
            seed_data = seed_data.sort_values("severity")
            ax.plot(
                seed_data["severity"],
                seed_data["log2_transfer_vs_baseline"],
                color=NETWORK_COLORS[network],
                alpha=0.25,
                linewidth=1.0,
            )
        median_data = (
            network_data.groupby("severity", as_index=False)[
                "log2_transfer_vs_baseline"
            ].median()
        )
        ax.plot(
            median_data["severity"],
            median_data["log2_transfer_vs_baseline"],
            color=NETWORK_COLORS[network],
            marker="o",
            linewidth=3.0,
            label=NETWORK_NAMES[network],
        )
    ax.axhline(0.0, color="black", linewidth=1.0, linestyle=":")
    ax.set(
        title=f"{probe} temporal probe",
        xlabel="AD-like perturbation strength",
        xticks=sorted(main_normalized_df["severity"].unique()),
    )
    ax.legend()
axes[0].set_ylabel("log2 target/A1 transfer relative to own baseline")
fig.suptitle("Full-field response across perturbation strengths")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "02_main_stage_curves.png", dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(7.8, 4.8))
positions = {"2Hz": 0, "5Hz": 1}
for probe, probe_data in primary_endpoint_df.groupby("probe"):
    x0 = positions[probe]
    offsets = np.linspace(-0.09, 0.09, len(probe_data))
    ax.scatter(
        x0 + offsets,
        probe_data["music_minus_speech_log2_change"],
        s=55,
        label=probe,
    )
    ax.hlines(
        probe_data["music_minus_speech_log2_change"].median(),
        x0 - 0.18,
        x0 + 0.18,
        color="black",
        linewidth=3,
    )
ax.axhline(0.0, color="black", linestyle=":")
ax.set(
    xticks=[0, 1],
    xticklabels=["2 Hz", "5 Hz"],
    ylabel="Music minus speech log2 change",
    title="High-endpoint contrast across numerical seeds",
)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "03_primary_endpoint_contrast.png", dpi=180)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.8), sharey=True)
analysis_order = [
    "Full regional perturbation",
    "A1 and primary targets locally fixed",
]
for ax, probe in zip(axes, PERIODIC_PROBES):
    subset = counterfactual_comparison_df[
        counterfactual_comparison_df["probe"] == probe
    ]
    for position, analysis in enumerate(analysis_order):
        values = subset[subset["analysis"] == analysis][
            "music_minus_speech_log2_change"
        ].to_numpy()
        offsets = np.linspace(-0.08, 0.08, len(values))
        ax.scatter(position + offsets, values, s=45)
        ax.hlines(
            np.median(values),
            position - 0.17,
            position + 0.17,
            color="black",
            linewidth=3,
        )
    ax.axhline(0.0, color="black", linestyle=":")
    ax.set(
        title=probe,
        xticks=[0, 1],
        xticklabels=["Full field", "Local fixed"],
        xlabel="Analysis",
    )
axes[0].set_ylabel("Music minus speech log2 change")
fig.suptitle("Local-dynamics counterfactual")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "04_local_dynamics_counterfactual.png", dpi=180)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.6), sharey=True)
for ax, probe in zip(axes, PERIODIC_PROBES):
    null_subset = matched_null_df[
        (matched_null_df["seed"] == main_seed)
        & (matched_null_df["probe"] == probe)
    ]["null_music_minus_speech"]
    observed_value = primary_endpoint_df[
        (primary_endpoint_df["seed"] == main_seed)
        & (primary_endpoint_df["probe"] == probe)
    ]["music_minus_speech_log2_change"].iloc[0]
    ax.hist(null_subset, bins=24, color="#9CA3AF", edgecolor="white")
    ax.axvline(
        observed_value,
        color="#B91C1C",
        linewidth=3,
        label="Observed proxy contrast",
    )
    ax.axvline(0.0, color="black", linestyle=":")
    ax.set(title=probe, xlabel="Matched-control contrast")
    ax.legend()
axes[0].set_ylabel("Matched control-set count")
fig.suptitle(f"Simulation-level matched null, numerical seed {main_seed}")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "05_matched_control_null.png", dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(9.0, 4.8))
scenario_labels = list(dict.fromkeys(sensitivity_endpoint_df["variant"]))
x_positions = np.arange(len(scenario_labels))
width = 0.28
for offset, probe in [(-width / 2, "2Hz"), (width / 2, "5Hz")]:
    values = [
        sensitivity_endpoint_df[
            (sensitivity_endpoint_df["variant"] == scenario)
            & (sensitivity_endpoint_df["probe"] == probe)
        ]["music_minus_speech_log2_change"].iloc[0]
        for scenario in scenario_labels
    ]
    ax.scatter(
        x_positions + offset,
        values,
        s=65,
        label=probe,
    )
ax.axhline(0.0, color="black", linestyle=":")
ax.set(
    xticks=x_positions,
    xticklabels=scenario_labels,
    ylabel="Music minus speech log2 change",
    title="Parameter sensitivity at the high endpoint",
)
ax.tick_params(axis="x", rotation=25)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "06_parameter_sensitivity.png", dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(7.8, 4.8))
for probe_index, probe in enumerate(PERIODIC_PROBES):
    shuffle_values = shuffle_contrast_df[
        shuffle_contrast_df["probe"] == probe
    ]["music_minus_speech_log2_change"].to_numpy()
    offsets = np.linspace(-0.09, 0.09, len(shuffle_values))
    ax.scatter(
        probe_index + offsets,
        shuffle_values,
        color="#6B7280",
        s=48,
        label="Spatial shuffles" if probe_index == 0 else None,
    )
    observed_value = observed_first_seed_df[
        observed_first_seed_df["probe"] == probe
    ]["observed_contrast"].iloc[0]
    ax.scatter(
        [probe_index],
        [observed_value],
        marker="*",
        s=190,
        color="#B91C1C",
        label="Observed placement" if probe_index == 0 else None,
    )
ax.axhline(0.0, color="black", linestyle=":")
ax.set(
    xticks=[0, 1],
    xticklabels=["2 Hz", "5 Hz"],
    ylabel="Music minus speech log2 change",
    title="Sensitivity to regional perturbation placement",
)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "07_spatial_placement_sensitivity.png", dpi=180)
plt.show()

# Secondary musical semantic-versus-episodic proxy overview.
MEMORY_NETWORK_COLORS = {
    "music_semantic_task_associated": "#6B46C1",
    "music_episodic_task_associated": "#0F766E",
}
MEMORY_NETWORK_NAMES = {
    "music_semantic_task_associated": "Semantic-task-associated proxy",
    "music_episodic_task_associated": "Episodic-task-associated proxy",
}

fig, axes = plt.subplots(2, 2, figsize=(13.0, 9.2))
for ax, probe in zip(axes[0], PERIODIC_PROBES):
    plot_data = main_normalized_df[
        main_normalized_df["probe"] == probe
    ]
    for network in MEMORY_NETWORK_COLORS:
        network_data = plot_data[
            plot_data["network"] == network
        ]
        for _, seed_data in network_data.groupby("seed"):
            seed_data = seed_data.sort_values("severity")
            ax.plot(
                seed_data["severity"],
                seed_data["log2_transfer_vs_baseline"],
                color=MEMORY_NETWORK_COLORS[network],
                alpha=0.25,
                linewidth=1.0,
            )
        median_data = (
            network_data.groupby("severity", as_index=False)[
                "log2_transfer_vs_baseline"
            ].median()
        )
        ax.plot(
            median_data["severity"],
            median_data["log2_transfer_vs_baseline"],
            color=MEMORY_NETWORK_COLORS[network],
            marker="o",
            linewidth=3.0,
            label=MEMORY_NETWORK_NAMES[network],
        )
    ax.axhline(0.0, color="black", linewidth=1.0, linestyle=":")
    ax.set(
        title=f"{probe} temporal probe",
        xlabel="AD-like perturbation strength",
        xticks=sorted(main_normalized_df["severity"].unique()),
    )
    ax.legend(fontsize=8)
axes[0, 0].set_ylabel("log2 target/A1 transfer relative to own baseline")

ax = axes[1, 0]
positions = {"2Hz": 0, "5Hz": 1}
for probe, probe_data in secondary_endpoint_df.groupby("probe"):
    x0 = positions[probe]
    offsets = np.linspace(-0.09, 0.09, len(probe_data))
    ax.scatter(
        x0 + offsets,
        probe_data["semantic_minus_episodic_log2_change"],
        s=55,
    )
    ax.hlines(
        probe_data["semantic_minus_episodic_log2_change"].median(),
        x0 - 0.18,
        x0 + 0.18,
        color="black",
        linewidth=3,
    )
ax.axhline(0.0, color="black", linestyle=":")
ax.set(
    xticks=[0, 1],
    xticklabels=["2 Hz", "5 Hz"],
    ylabel="Semantic-associated minus episodic-associated log2 change",
    title="High-endpoint secondary contrast",
)

ax = axes[1, 1]
memory_analysis_order = [
    "Full regional perturbation",
    "A1 and memory-proxy targets locally fixed",
]
memory_probe_colors = {"2Hz": "#7C3AED", "5Hz": "#0F766E"}
for probe_offset, probe in zip((-0.08, 0.08), PERIODIC_PROBES):
    subset = memory_counterfactual_comparison_df[
        memory_counterfactual_comparison_df["probe"] == probe
    ]
    for position, analysis in enumerate(memory_analysis_order):
        values = subset[subset["analysis"] == analysis][
            "semantic_minus_episodic_log2_change"
        ].to_numpy()
        offsets = np.linspace(-0.035, 0.035, len(values))
        ax.scatter(
            position + probe_offset + offsets,
            values,
            s=42,
            color=memory_probe_colors[probe],
            label=probe if position == 0 else None,
        )
        ax.hlines(
            np.median(values),
            position + probe_offset - 0.07,
            position + probe_offset + 0.07,
            color=memory_probe_colors[probe],
            linewidth=3,
        )
ax.axhline(0.0, color="black", linestyle=":")
ax.set(
    xticks=[0, 1],
    xticklabels=["Full field", "Memory targets local-fixed"],
    ylabel="Semantic-associated minus episodic-associated log2 change",
    title="Secondary local-dynamics counterfactual",
)
ax.legend()

fig.suptitle(
    "Secondary musical-memory task-associated proxy analysis"
)
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "08_semantic_episodic_secondary_analysis.png",
    dpi=180,
)
plt.show()


fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.6), sharey=True)
for ax, probe in zip(axes, PERIODIC_PROBES):
    null_subset = memory_matched_null_df[
        (memory_matched_null_df["seed"] == main_seed)
        & (memory_matched_null_df["probe"] == probe)
    ]["null_semantic_minus_episodic"]
    observed_value = secondary_endpoint_df[
        (secondary_endpoint_df["seed"] == main_seed)
        & (secondary_endpoint_df["probe"] == probe)
    ]["semantic_minus_episodic_log2_change"].iloc[0]
    ax.hist(
        null_subset,
        bins=24,
        color="#9CA3AF",
        edgecolor="white",
    )
    ax.axvline(
        observed_value,
        color="#6B21A8",
        linewidth=3,
        label="Observed secondary contrast",
    )
    ax.axvline(0.0, color="black", linestyle=":")
    ax.set(title=probe, xlabel="Matched-control contrast")
    ax.legend()
axes[0].set_ylabel("Matched control-set count")
fig.suptitle(
    f"Secondary matched null, numerical seed {main_seed}"
)
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "09_semantic_episodic_matched_null.png",
    dpi=180,
)
plt.show()


fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.8))
ax = axes[0]
scenario_labels = list(
    dict.fromkeys(sensitivity_endpoint_df["variant"])
)
x_positions = np.arange(len(scenario_labels))
width = 0.28
for offset, probe in [(-width / 2, "2Hz"), (width / 2, "5Hz")]:
    values = [
        sensitivity_endpoint_df[
            (sensitivity_endpoint_df["variant"] == scenario)
            & (sensitivity_endpoint_df["probe"] == probe)
        ]["semantic_minus_episodic_log2_change"].iloc[0]
        for scenario in scenario_labels
    ]
    ax.scatter(
        x_positions + offset,
        values,
        s=65,
        label=probe,
    )
ax.axhline(0.0, color="black", linestyle=":")
ax.set(
    xticks=x_positions,
    xticklabels=scenario_labels,
    ylabel="Semantic-associated minus episodic-associated log2 change",
    title="Secondary parameter sensitivity",
)
ax.tick_params(axis="x", rotation=25)
ax.legend()

ax = axes[1]
for probe_index, probe in enumerate(PERIODIC_PROBES):
    shuffle_values = shuffle_contrast_df[
        shuffle_contrast_df["probe"] == probe
    ]["semantic_minus_episodic_log2_change"].to_numpy()
    offsets = np.linspace(-0.09, 0.09, len(shuffle_values))
    ax.scatter(
        probe_index + offsets,
        shuffle_values,
        color="#6B7280",
        s=48,
        label="Spatial shuffles" if probe_index == 0 else None,
    )
    observed_value = memory_observed_first_seed_df[
        memory_observed_first_seed_df["probe"] == probe
    ]["observed_contrast"].iloc[0]
    ax.scatter(
        [probe_index],
        [observed_value],
        marker="*",
        s=190,
        color="#6B21A8",
        label="Observed placement" if probe_index == 0 else None,
    )
ax.axhline(0.0, color="black", linestyle=":")
ax.set(
    xticks=[0, 1],
    xticklabels=["2 Hz", "5 Hz"],
    ylabel="Semantic-associated minus episodic-associated log2 change",
    title="Secondary spatial-placement sensitivity",
)
ax.legend()
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "10_semantic_episodic_robustness.png",
    dpi=180,
)
plt.show()


## 18. Save all results and provenance

CSV files preserve the node-level and network-level outputs. The run manifest records the worker process and deterministic job ordinal for every TVB simulation. The JSON metadata records sources, parameters, parallel execution settings, parcel definitions, coordinate-to-parcel mapping provenance, prespecified contrasts, and interpretation limits. The final ZIP can be downloaded from Colab.

Primary and secondary contrasts are saved in separate named tables even though they are calculated from the same 379-node simulations.

In [ ]:
sensitivity_manifest_df = (
    pd.concat(sensitivity_manifest_frames, ignore_index=True)
    if sensitivity_manifest_frames
    else pd.DataFrame()
)
shuffle_manifest_df = pd.concat(shuffle_manifest_frames, ignore_index=True)
all_manifest_df = pd.concat(
    [
        main_manifest_df,
        local_fixed_manifest_df,
        sensitivity_manifest_df,
        shuffle_manifest_df,
        dt_reference_manifest_df,
    ],
    ignore_index=True,
)

output_tables = {
    "source_manifest.csv": source_manifest_df,
    "data_quality_checks.csv": data_quality_df,
    "roi_definitions.csv": roi_definition_df,
    "music_memory_peak_mapping.csv": music_memory_peak_mapping_df,
    "roi_pathology_values.csv": roi_pathology_df,
    "pathology_summary.csv": pathology_summary_df,
    "baseline_coupling_calibration.csv": calibration_df,
    "main_node_metrics.csv": main_node_df,
    "main_network_metrics.csv": main_network_df,
    "main_network_metrics_normalized.csv": main_normalized_df,
    "main_music_minus_speech_contrasts.csv": main_contrast_df,
    "main_semantic_minus_episodic_contrasts.csv": (
        secondary_main_contrast_df
    ),
    "main_stage_summary.csv": main_stage_summary_df,
    "local_fixed_node_metrics.csv": local_fixed_node_df,
    "local_fixed_network_metrics.csv": local_fixed_network_df,
    "local_fixed_contrasts.csv": local_fixed_contrast_df,
    "memory_counterfactual_comparison.csv": (
        memory_counterfactual_comparison_df
    ),
    "matched_control_sets.csv": matched_sets_df,
    "matched_control_null_metrics.csv": matched_null_df,
    "matched_control_null_summary.csv": matched_null_summary_df,
    "memory_matched_control_sets.csv": memory_matched_sets_df,
    "memory_matched_control_null_metrics.csv": memory_matched_null_df,
    "memory_matched_control_null_summary.csv": (
        memory_matched_null_summary_df
    ),
    "sensitivity_network_metrics.csv": sensitivity_network_df,
    "sensitivity_contrasts.csv": sensitivity_contrast_df,
    "spatial_shuffle_network_metrics.csv": shuffle_network_df,
    "spatial_shuffle_contrasts.csv": shuffle_contrast_df,
    "spatial_shuffle_summary.csv": shuffle_summary_df,
    "memory_spatial_shuffle_summary.csv": memory_shuffle_summary_df,
    "integration_step_check.csv": dt_convergence_df,
    "run_manifest.csv": all_manifest_df,
}
for filename, frame in output_tables.items():
    frame.to_csv(RESULTS_DIR / filename, index=False)

metadata = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "run_mode": RUN_MODE,
    "research_question": (
        "How does increasing AD-like amyloid-linked inhibitory dysfunction "
        "affect stimulus-evoked transmission from bilateral primary auditory "
        "cortex to music-associated versus speech-associated cortical proxy subnetworks?"
    ),
    "source_commits": {
        "educase": EDUCASE_COMMIT,
        "adni_tvb_pipeline": PIPELINE_COMMIT,
    },
    "source_hashes": {
        row.source: row.sha256
        for row in source_manifest_df.itertuples(index=False)
    },
    "execution": {
        "parallel_backend": PARALLEL_BACKEND,
        "joblib_version": joblib.__version__,
        "cpu_allocation_sources": CPU_ALLOCATION_SOURCES,
        "available_cpu_count": AVAILABLE_CPU_COUNT,
        "requested_worker_processes": REQUESTED_PARALLEL_WORKERS,
        "configured_worker_processes": PARALLEL_WORKERS,
        "native_threads_per_worker": NATIVE_THREADS_PER_WORKER,
        "native_thread_environment": {
            variable_name: os.environ.get(variable_name)
            for variable_name in NATIVE_THREAD_ENVIRONMENT_VARIABLES
        },
        "job_granularity": (
            "one condition-seed block: one matched control plus "
            "all requested probes"
        ),
        "observed_worker_pids": sorted(
            int(pid)
            for pid in all_manifest_df["worker_pid"].dropna().unique()
        ),
        "observed_worker_processes": int(
            all_manifest_df["worker_pid"].nunique()
        ),
        "worker_override_environment_variable": "RISE_N_WORKERS",
        "loky_psutil_memory_monitor_disabled": (
            os.environ.get("RISE_DISABLE_LOKY_PSUTIL", "").strip()
            == "1"
        ),
    },
    "model": {
        "regions": N_REGIONS,
        "global_coupling": MAIN_GLOBAL_COUPLING,
        "input_peak_per_ms": MAIN_INPUT_PEAK_PER_MS,
        "dt_ms": MAIN_DT_MS,
        "monitor_period_ms": MONITOR_PERIOD_MS,
        "simulation_ms": SIMULATION_MS,
        "stimulus_onset_ms": STIMULUS_ONSET_MS,
        "delays": "all zero, following the public educational model",
        "probes": list(PROBES),
        "numerical_seeds": CFG["seeds"],
    },
    "parcels": {
        "primary_music_proxy": list(MUSIC_LABELS),
        "primary_speech_proxy": list(SPEECH_LABELS),
        "secondary_semantic_task_associated_proxy": list(
            SEMANTIC_MEMORY_LABELS
        ),
        "secondary_episodic_task_associated_proxy": list(
            EPISODIC_MEMORY_LABELS
        ),
        "secondary_mapping_source": {
            "paper": (
                "Platel et al. 2003, NeuroImage 20:244-256, "
                "doi:10.1016/S1053-8119(03)00287-8"
            ),
            "coordinate_space": "SPM99 coordinates reported by paper",
            "atlas_reference": (
                "volumetric HCP-MMP1 reference; parcel labels validated "
                "against this notebook's 379-entry order"
            ),
            "mapping_table": "music_memory_peak_mapping.csv",
        },
        "roi_groups": {
            group_name: list(specification["labels"])
            for group_name, specification in ROI_GROUPS.items()
        },
    },
    "primary_metric": (
        "network harmonic amplitude divided by bilateral A1 harmonic amplitude, "
        "then log2-normalized to the same network's baseline"
    ),
    "primary_contrast": (
        "music log2 change minus speech log2 change"
    ),
    "secondary_contrast": (
        "semantic-task-associated log2 change minus "
        "episodic-task-associated log2 change"
    ),
    "secondary_analysis_status": (
        "prespecified secondary mechanistic analysis; not a memory task"
    ),
    "interpretation_limits": [
        "The public amyloid endpoint is an artificial surrogate, not patient data.",
        "The model includes amyloid-linked inhibition but not tau, atrophy, synapse loss, inflammation, vascular disease, or structural degeneration.",
        "The connectome is one averaged healthy structural matrix for every condition.",
        "All interregional delays are zero.",
        "The 2 Hz and 5 Hz inputs are temporal probes, not literal music and speech.",
        "The parcel groups are approximate proxy subnetworks.",
        "The model contains no memory encoding or retrieval mechanism.",
        "Semantic- and episodic-task-associated parcels are operational proxy sets, not proven separate pathways.",
        "The periodic probes contain no familiarity, encoding, delay, or recognition manipulation.",
        "Coordinate-to-parcel mapping is approximate because the source PET study and HCP-MMP atlas use different parcellation and registration frameworks.",
        "Numerical seeds and matched control sets are not human subjects.",
        "The HCP-MMP atlas contains PBelt but not separate rostral and caudal parabelt parcels.",
        "TA2/STGa are a gross planum-polare proxy, not a voxelwise music-selective functional localizer.",
        "6ma/24dd are parcel approximations of the Jacobsen musical-memory regions, not a direct surface-overlap mapping.",
    ],
}
(RESULTS_DIR / "experiment_metadata.json").write_text(
    json.dumps(metadata, indent=2)
)

archive_base = WORK_DIR / f"RISE_TVB379_results_{RUN_MODE}"
archive_path = shutil.make_archive(
    str(archive_base), "zip", root_dir=RESULTS_DIR
)
print("Saved result archive:", archive_path)
print("Total TVB simulations recorded:", len(all_manifest_df))
print(
    "Total recorded TVB wall time, minutes:",
    round(all_manifest_df["wall_seconds"].sum() / 60.0, 2),
)

if DOWNLOAD_RESULTS_AT_END and IS_COLAB:
    from google.colab import files
    files.download(archive_path)

## 19. How to interpret the final result

### Primary result

Read the music-versus-speech result in this order:

1. **Main stage curves:** determine how each proxy network's A1-normalized transfer changes from its own baseline.
2. **Primary contrast:** check whether the music-minus-speech direction is consistent across 2 Hz, 5 Hz, and numerical seeds.
3. **Primary local-dynamics counterfactual:** determine whether the contrast remains when A1 and primary-target \(b\) values are held at baseline.
4. **Primary matched controls:** determine whether the declared contrast is unusual relative to size-, topology-, and pathology-matched parcel groups.
5. **Parameter and spatial sensitivity:** determine whether the direction survives coupling, input, and perturbation-placement changes.

A careful supporting statement would be:

> Within this 379-region mechanistic simulation, the predeclared music-associated proxy showed a more favorable baseline-normalized transfer change than the speech-associated proxy under the tested AD-like amyloid-linked inhibitory perturbation, and the direction was assessed against local-dynamics, topology, parameter, and spatial-placement controls.

### Secondary result

Read the semantic-associated-versus-episodic-associated result separately:

1. Check both proxy trajectories rather than only their difference.
2. Require consistency across 2 Hz, 5 Hz, and numerical seeds.
3. Check the dedicated musical-memory-target local-dynamics counterfactual.
4. Compare the secondary contrast with its own 5-versus-4-parcel matched null.
5. Inspect secondary parameter and spatial-placement sensitivity.

A careful secondary statement would be:

> Transmission into parcel sets associated with semantic-versus-episodic musical-memory tasks showed [similar/different] baseline-normalized trajectories under the modeled perturbation.

Do not replace `associated with musical-memory tasks` with `semantic memory pathway`, `episodic memory pathway`, or `preserved memory`. The notebook has no cognitive task or memory state. A secondary result can refine the primary mechanism; it cannot rescue a failed primary music-versus-speech hypothesis.

## 20. Known limitations

1. The downloadable amyloid data are artificial surrogates, so the experiment cannot support clinical group inference.
2. Only one proposed AD mechanism is modeled: amyloid-linked change in the inhibitory time constant.
3. Tau, atrophy, synaptic loss, neuroinflammation, vascular disease, and white-matter degeneration are absent.
4. The amyloid-to-\(b\) mapping is a modeling hypothesis, not an established biological law.
5. Every condition uses the same averaged healthy structural connectome.
6. Interregional delays are zero, so latency and realistic phase claims are excluded.
7. Diffusion-MRI tractography is incomplete, can contain false connections, and does not establish biological direction.
8. Whole Glasser parcels cannot isolate fine-grained music-, speech-, semantic-, or episodic-selective neural populations.
9. The 2 Hz and 5 Hz probes omit pitch, melody, timbre, language, meaning, familiarity, emotion, encoding, delay, and recognition.
10. The model has no memory process, so it does not test musical-memory preservation.
11. The semantic and episodic proxy sets are mapped from activation peaks in a small PET study and do not capture full activation clusters.
12. The source PET coordinate space and volumetric HCP-MMP reference are not identical; the mapping is documented but approximate.
13. Semantic and episodic retrieval may rely substantially on shared circuitry. The secondary split is an operational task contrast, not proof of separate anatomical pathways.
14. Results may still depend on local dynamics, network topology, coupling, input strength, initial conditions, and the artificial spatial perturbation map.
15. A finer 379-region atlas improves anatomical resolution but does not make the neural-mass model cellular or cognitively complete.

## 21. Primary references

- Stefanovski, L., et al. (2019). Linking molecular pathways and large-scale computational modeling to assess candidate disease mechanisms and pharmacodynamics in Alzheimer's disease. *Frontiers in Computational Neuroscience, 13*, 54. https://doi.org/10.3389/fncom.2019.00054
- BrainModes. *TVB Educase AD molecular pathways*, public surrogate notebook and data. https://github.com/BrainModes/TVB_EducaseAD_molecular_pathways_TVB
- BrainModes. *ADNI-TVB pipeline*, 379-region label order. https://github.com/BrainModes/ADNI-TVB-pipeline
- Glasser, M. F., et al. (2016). A multi-modal parcellation of human cerebral cortex. *Nature, 536*, 171-178. https://doi.org/10.1038/nature18933
- Ding, N., et al. (2017). Temporal modulations in speech and music. *Neuroscience & Biobehavioral Reviews, 81*, 181-187. https://doi.org/10.1016/j.neubiorev.2017.02.011
- Jacobsen, J.-H., et al. (2015). Why musical memory can be preserved in advanced Alzheimer's disease. *Brain, 138*, 2438-2450. https://doi.org/10.1093/brain/awv135
- Platel, H., Baron, J.-C., Desgranges, B., Bernard, F., & Eustache, F. (2003). Semantic and episodic memory of music are subserved by distinct neural networks. *NeuroImage, 20*, 244-256. https://doi.org/10.1016/S1053-8119(03)00287-8
- Slattery, C. F., et al. (2019). The functional neuroanatomy of musical memory in Alzheimer's disease. *Cortex, 115*, 357-370. https://doi.org/10.1016/j.cortex.2019.02.003
- Tibon, R., et al. (2026). Neural activations and representations during episodic versus semantic memory retrieval. *Nature Human Behaviour, 10*, 803-821. https://doi.org/10.1038/s41562-025-02390-4
- Hickok, G., & Poeppel, D. (2007). The cortical organization of speech processing. *Nature Reviews Neuroscience, 8*, 393-402. https://doi.org/10.1038/nrn2113
- Norman-Haignere, S., Kanwisher, N. G., & McDermott, J. H. (2015). Distinct cortical pathways for music and speech revealed by hypothesis-free voxel decomposition. *Neuron, 88*, 1281-1296. https://doi.org/10.1016/j.neuron.2015.11.035
- Maier-Hein, K. H., et al. (2017). The challenge of mapping the human connectome based on diffusion tractography. *Nature Communications, 8*, 1349. https://doi.org/10.1038/s41467-017-01285-x